In [1]:
# ============================================================
# V2 — Cell 1
# Density-matrix core + mixed-state concurrence
#
# Purpose:
#   1. Reproduce the V1 ideal Bell-like circuit exactly.
#   2. Introduce density matrices as the V2 state representation.
#   3. Add Wootters concurrence for arbitrary two-qubit states.
#   4. Establish unit tests before introducing noise.
#
# Convention:
#   alpha here is the same circuit angle called theta in the brief.
# ============================================================

import numpy as np


# ------------------------------------------------------------
# 1. Single-qubit basis states and operators
# ------------------------------------------------------------

ket0 = np.array([1.0, 0.0], dtype=complex)
ket1 = np.array([0.0, 1.0], dtype=complex)

I2 = np.eye(2, dtype=complex)

X = np.array([
    [0, 1],
    [1, 0]
], dtype=complex)

Y = np.array([
    [0, -1j],
    [1j, 0]
], dtype=complex)

Z = np.array([
    [1,  0],
    [0, -1]
], dtype=complex)


# ------------------------------------------------------------
# 2. Two-qubit basis and CNOT
#
# Basis ordering:
#   |00>, |01>, |10>, |11>
#
# First qubit = control
# Second qubit = target
# ------------------------------------------------------------

ket00 = np.kron(ket0, ket0)

CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
], dtype=complex)

YY = np.kron(Y, Y)


# ------------------------------------------------------------
# 3. Basic state utilities
# ------------------------------------------------------------

def normalize_ket(psi, atol=1e-15):
    """
    Return a normalized copy of a state vector.
    """
    psi = np.asarray(psi, dtype=complex).reshape(-1)

    norm = np.linalg.norm(psi)

    if norm < atol:
        raise ValueError("Cannot normalize the zero vector.")

    return psi / norm


def ket_to_rho(psi):
    """
    Convert |psi> to rho = |psi><psi|.
    """
    psi = normalize_ket(psi)
    return np.outer(psi, psi.conj())


def purity(rho):
    """
    Purity Tr(rho^2).

    Pure state:  1
    Mixed state: < 1
    """
    rho = np.asarray(rho, dtype=complex)

    return float(
        np.real(
            np.trace(rho @ rho)
        )
    )


# ------------------------------------------------------------
# 4. V1 gate definitions
# ------------------------------------------------------------

def Ry(alpha):
    """
    Single-qubit Y rotation.

    Ry(alpha) =
        [[ cos(alpha/2), -sin(alpha/2)],
         [ sin(alpha/2),  cos(alpha/2)]]
    """
    c = np.cos(alpha / 2.0)
    s = np.sin(alpha / 2.0)

    return np.array([
        [c, -s],
        [s,  c]
    ], dtype=complex)


def apply_unitary(rho, U):
    """
    Density-matrix unitary evolution:

        rho -> U rho U^\dagger
    """
    rho = np.asarray(rho, dtype=complex)
    U = np.asarray(U, dtype=complex)

    return U @ rho @ U.conj().T


# ------------------------------------------------------------
# 5. V1 reference state
# ------------------------------------------------------------

def bell_like_ket(alpha):
    """
    Analytic V1 reference state:

        |psi(alpha)> =
            cos(alpha/2)|00>
            + sin(alpha/2)|11>
    """
    return np.array([
        np.cos(alpha / 2.0),
        0.0,
        0.0,
        np.sin(alpha / 2.0)
    ], dtype=complex)


# ------------------------------------------------------------
# 6. V2 density-matrix implementation of the same circuit
# ------------------------------------------------------------

def prepare_ideal_density_matrix(alpha):
    """
    Prepare the V1 Bell-like state using density matrices.

    Circuit:

        |00>
          |
        Ry(alpha) on qubit 0
          |
        CNOT(0 -> 1)
          |
        rho_out

    This is the zero-noise baseline for V2.
    """

    # Initial pure state
    rho = ket_to_rho(ket00)

    # Ry on first qubit
    U_ry = np.kron(
        Ry(alpha),
        I2
    )

    rho = apply_unitary(
        rho,
        U_ry
    )

    # Entangling gate
    rho = apply_unitary(
        rho,
        CNOT
    )

    return rho


# ------------------------------------------------------------
# 7. Mixed-state concurrence
#
# Wootters concurrence:
#
#   rho_tilde =
#       (Y⊗Y) rho* (Y⊗Y)
#
# Let lambda_i be the decreasing square roots of the
# eigenvalues of rho rho_tilde.
#
#   C = max(0, lambda1-lambda2-lambda3-lambda4)
# ------------------------------------------------------------

def concurrence_mixed(rho):
    """
    Wootters concurrence for an arbitrary two-qubit
    density matrix.

    Returns:
        C in [0, 1]
    """

    rho = np.asarray(
        rho,
        dtype=complex
    )

    if rho.shape != (4, 4):
        raise ValueError(
            "rho must be a 4x4 two-qubit density matrix."
        )

    rho_tilde = (
        YY
        @ rho.conj()
        @ YY
    )

    eigvals = np.linalg.eigvals(
        rho @ rho_tilde
    )

    # Exact theory gives real, non-negative eigenvalues.
    # Clean up small numerical roundoff.
    eigvals = np.real_if_close(
        eigvals,
        tol=1000
    )

    eigvals = np.real(
        eigvals
    )

    eigvals = np.clip(
        eigvals,
        0.0,
        None
    )

    lambdas = np.sort(
        np.sqrt(eigvals)
    )[::-1]

    C = (
        lambdas[0]
        - np.sum(lambdas[1:])
    )

    return float(
        np.clip(
            C,
            0.0,
            1.0
        )
    )


# ------------------------------------------------------------
# 8. Fidelity with a pure target
#
# For pure target |psi_t>:
#
#   F = <psi_t| rho |psi_t>
# ------------------------------------------------------------

def fidelity_with_pure_target(rho, psi_target):
    """
    Fidelity of a possibly mixed state rho with a
    pure target state |psi_target>.
    """

    rho = np.asarray(
        rho,
        dtype=complex
    )

    psi_target = normalize_ket(
        psi_target
    )

    F = np.vdot(
        psi_target,
        rho @ psi_target
    )

    return float(
        np.clip(
            np.real_if_close(F).real,
            0.0,
            1.0
        )
    )


# ------------------------------------------------------------
# 9. V1 target-state construction
# ------------------------------------------------------------

def target_from_concurrence(target_concurrence):
    """
    Construct the same principal-branch target used in V1.

        alpha_target = arcsin(C_target)

    with

        0 <= alpha_target <= pi/2
    """

    C = float(
        target_concurrence
    )

    if not (0.0 <= C <= 1.0):
        raise ValueError(
            "target_concurrence must lie in [0, 1]."
        )

    alpha_target = np.arcsin(C)

    psi_target = bell_like_ket(
        alpha_target
    )

    return (
        alpha_target,
        psi_target
    )


# ------------------------------------------------------------
# 10. Density-matrix validity check
# ------------------------------------------------------------

def assert_density_matrix(rho, atol=1e-12):
    """
    Verify the basic physical requirements:

        rho = rho^\dagger
        Tr(rho) = 1
        rho >= 0
    """

    rho = np.asarray(
        rho,
        dtype=complex
    )

    assert rho.shape == (4, 4)

    assert np.allclose(
        rho,
        rho.conj().T,
        atol=atol
    ), "rho is not Hermitian"

    assert np.isclose(
        np.trace(rho),
        1.0,
        atol=atol
    ), "Tr(rho) != 1"

    eigvals = np.linalg.eigvalsh(
        rho
    )

    assert np.min(eigvals) >= -atol, (
        f"rho is not positive semidefinite: {eigvals}"
    )


# ------------------------------------------------------------
# 11. CELL 1 UNIT TESTS
# ------------------------------------------------------------

def run_cell1_unit_tests(verbose=True):

    # --------------------------------------------------------
    # Test 1:
    # Density-matrix circuit reproduces V1 exactly
    # --------------------------------------------------------

    test_angles = [
        0.0,
        0.23,
        0.71,
        np.pi / 2,
        1.91
    ]

    for alpha in test_angles:

        rho = prepare_ideal_density_matrix(
            alpha
        )

        rho_reference = ket_to_rho(
            bell_like_ket(alpha)
        )

        assert_density_matrix(
            rho
        )

        assert np.allclose(
            rho,
            rho_reference,
            atol=1e-12
        ), (
            "Density-matrix circuit disagrees "
            f"with V1 at alpha={alpha}"
        )

        # ----------------------------------------------------
        # Test 2:
        # Ideal state remains pure
        # ----------------------------------------------------

        assert np.isclose(
            purity(rho),
            1.0,
            atol=1e-12
        ), (
            f"Purity != 1 at alpha={alpha}"
        )

        # ----------------------------------------------------
        # Test 3:
        # Mixed-state concurrence reproduces
        #
        #     C(alpha) = |sin(alpha)|
        # ----------------------------------------------------

        C_numeric = concurrence_mixed(
            rho
        )

        C_exact = abs(
            np.sin(alpha)
        )

        assert np.isclose(
            C_numeric,
            C_exact,
            atol=1e-8
        ), (
            "Concurrence mismatch at "
            f"alpha={alpha}: "
            f"{C_numeric} vs {C_exact}"
        )

    # --------------------------------------------------------
    # Test 4:
    # Known concurrence endpoints
    # --------------------------------------------------------

    C_product = concurrence_mixed(
        prepare_ideal_density_matrix(0.0)
    )

    C_bell = concurrence_mixed(
        prepare_ideal_density_matrix(
            np.pi / 2
        )
    )

    assert np.isclose(
        C_product,
        0.0,
        atol=1e-12
    )

    assert np.isclose(
        C_bell,
        1.0,
        atol=1e-8
    )

    # --------------------------------------------------------
    # Test 5:
    # V1 target construction gives unit fidelity
    # --------------------------------------------------------

    for C_target in [
        0.0,
        0.25,
        0.60,
        1.0
    ]:

        (
            alpha_target,
            psi_target
        ) = target_from_concurrence(
            C_target
        )

        rho_target = (
            prepare_ideal_density_matrix(
                alpha_target
            )
        )

        F = fidelity_with_pure_target(
            rho_target,
            psi_target
        )

        assert np.isclose(
            abs(np.sin(alpha_target)),
            C_target,
            atol=1e-12
        )

        assert np.isclose(
            F,
            1.0,
            atol=1e-12
        )

    if verbose:

        print("Cell 1 unit tests: PASS")
        print(
            "  ✓ density-matrix circuit reproduces V1"
        )
        print(
            "  ✓ trace / Hermiticity / positivity pass"
        )
        print(
            "  ✓ ideal state purity = 1"
        )
        print(
            "  ✓ Wootters concurrence reproduces |sin(alpha)|"
        )
        print(
            "  ✓ target-state fidelity reproduces F = 1"
        )


# Run tests immediately
run_cell1_unit_tests()

Cell 1 unit tests: PASS
  ✓ density-matrix circuit reproduces V1
  ✓ trace / Hermiticity / positivity pass
  ✓ ideal state purity = 1
  ✓ Wootters concurrence reproduces |sin(alpha)|
  ✓ target-state fidelity reproduces F = 1


In [2]:
# ============================================================
# V2 — Cell 2
# Basic quantum noise channels + unit tests
#
# Requires Cell 1.
#
# Adds:
#   1. Amplitude damping / relaxation
#   2. Pure dephasing / phase damping
#   3. Single-qubit depolarizing noise
#   4. Local application to either qubit
#
# IMPORTANT:
#   These channels are tested independently here.
#   We do NOT yet insert them into the Bell-state circuit.
# ============================================================


# ------------------------------------------------------------
# 1. Parameter validation
# ------------------------------------------------------------

def _check_probability(p, name="p"):
    """
    Require a probability-like parameter in [0, 1].
    """
    p = float(p)

    if not (0.0 <= p <= 1.0):
        raise ValueError(
            f"{name} must lie in [0, 1]."
        )

    return p


# ------------------------------------------------------------
# 2. Kraus completeness check
#
# A trace-preserving channel must satisfy
#
#     sum_k K_k^\dagger K_k = I
# ------------------------------------------------------------

def kraus_completeness(kraus_ops):
    """
    Return sum_k K_k^\dagger K_k.
    """
    total = np.zeros(
        (2, 2),
        dtype=complex
    )

    for K in kraus_ops:
        K = np.asarray(
            K,
            dtype=complex
        )

        total += K.conj().T @ K

    return total


def assert_cptp_kraus(kraus_ops, atol=1e-12):
    """
    Check the trace-preserving Kraus completeness relation.
    """

    total = kraus_completeness(
        kraus_ops
    )

    assert np.allclose(
        total,
        I2,
        atol=atol
    ), (
        "Kraus operators do not satisfy "
        "sum K†K = I."
    )


# ------------------------------------------------------------
# 3. Amplitude damping
#
# gamma = probability of |1> -> |0> relaxation
#
# K0 = [[1, 0],
#       [0, sqrt(1-gamma)]]
#
# K1 = [[0, sqrt(gamma)],
#       [0, 0]]
#
# gamma = 0 : identity
# gamma = 1 : complete relaxation
# ------------------------------------------------------------

def amplitude_damping_kraus(gamma):
    """
    Single-qubit amplitude-damping channel.
    """

    gamma = _check_probability(
        gamma,
        "gamma"
    )

    K0 = np.array([
        [1.0, 0.0],
        [0.0, np.sqrt(1.0 - gamma)]
    ], dtype=complex)

    K1 = np.array([
        [0.0, np.sqrt(gamma)],
        [0.0, 0.0]
    ], dtype=complex)

    return [K0, K1]


# ------------------------------------------------------------
# 4. Pure dephasing / phase damping
#
# Definition used here:
#
#   off-diagonal coherence -> (1-p) times coherence
#   populations unchanged
#
# p = 0 : identity
# p = 1 : complete dephasing
#
# Kraus representation:
#
# K0 = sqrt(1-p) I
# K1 = sqrt(p) |0><0|
# K2 = sqrt(p) |1><1|
# ------------------------------------------------------------

def phase_damping_kraus(p):
    """
    Pure dephasing channel.

    Under this convention, a one-qubit coherence
    rho_01 is multiplied by (1-p).
    """

    p = _check_probability(
        p,
        "p"
    )

    P0 = np.array([
        [1.0, 0.0],
        [0.0, 0.0]
    ], dtype=complex)

    P1 = np.array([
        [0.0, 0.0],
        [0.0, 1.0]
    ], dtype=complex)

    K0 = np.sqrt(1.0 - p) * I2
    K1 = np.sqrt(p) * P0
    K2 = np.sqrt(p) * P1

    return [K0, K1, K2]


# ------------------------------------------------------------
# 5. Depolarizing channel
#
# Convention:
#
#   E(rho) = (1-p) rho + p I/2
#
# for a single isolated qubit.
#
# Equivalent Pauli Kraus representation:
#
# K0 = sqrt(1 - 3p/4) I
# K1 = sqrt(p/4) X
# K2 = sqrt(p/4) Y
# K3 = sqrt(p/4) Z
#
# p = 0 : identity
# p = 1 : completely depolarized single qubit
# ------------------------------------------------------------

def depolarizing_kraus(p):
    """
    Standard single-qubit depolarizing channel.
    """

    p = _check_probability(
        p,
        "p"
    )

    K0 = (
        np.sqrt(1.0 - 3.0 * p / 4.0)
        * I2
    )

    scale = np.sqrt(
        p / 4.0
    )

    K1 = scale * X
    K2 = scale * Y
    K3 = scale * Z

    return [K0, K1, K2, K3]


# ------------------------------------------------------------
# 6. Apply a single-qubit Kraus channel to a two-qubit state
#
# qubit = 0 -> first / left qubit
# qubit = 1 -> second / right qubit
# ------------------------------------------------------------

def apply_local_kraus_channel(
    rho,
    kraus_ops,
    qubit
):
    """
    Apply a one-qubit CPTP channel to either qubit
    of a two-qubit density matrix.
    """

    rho = np.asarray(
        rho,
        dtype=complex
    )

    if rho.shape != (4, 4):
        raise ValueError(
            "rho must be a 4x4 two-qubit density matrix."
        )

    if qubit not in (0, 1):
        raise ValueError(
            "qubit must be 0 or 1."
        )

    rho_out = np.zeros_like(
        rho,
        dtype=complex
    )

    for K in kraus_ops:

        K = np.asarray(
            K,
            dtype=complex
        )

        if K.shape != (2, 2):
            raise ValueError(
                "Each local Kraus operator must be 2x2."
            )

        if qubit == 0:
            K_full = np.kron(
                K,
                I2
            )

        else:
            K_full = np.kron(
                I2,
                K
            )

        rho_out += (
            K_full
            @ rho
            @ K_full.conj().T
        )

    return rho_out


# ------------------------------------------------------------
# 7. Convenience wrappers
# ------------------------------------------------------------

def apply_amplitude_damping(
    rho,
    gamma,
    qubit
):
    return apply_local_kraus_channel(
        rho,
        amplitude_damping_kraus(gamma),
        qubit
    )


def apply_phase_damping(
    rho,
    p,
    qubit
):
    return apply_local_kraus_channel(
        rho,
        phase_damping_kraus(p),
        qubit
    )


def apply_depolarizing(
    rho,
    p,
    qubit
):
    return apply_local_kraus_channel(
        rho,
        depolarizing_kraus(p),
        qubit
    )


# ------------------------------------------------------------
# 8. Cell 2 unit tests
# ------------------------------------------------------------

def run_cell2_unit_tests(verbose=True):

    # Bell state from the validated Cell 1 core
    rho_bell = prepare_ideal_density_matrix(
        np.pi / 2
    )

    # --------------------------------------------------------
    # Test 1:
    # All Kraus sets satisfy completeness
    # --------------------------------------------------------

    for gamma in [
        0.0,
        0.17,
        0.63,
        1.0
    ]:
        assert_cptp_kraus(
            amplitude_damping_kraus(gamma)
        )

    for p in [
        0.0,
        0.17,
        0.63,
        1.0
    ]:
        assert_cptp_kraus(
            phase_damping_kraus(p)
        )

        assert_cptp_kraus(
            depolarizing_kraus(p)
        )

    # --------------------------------------------------------
    # Test 2:
    # Zero noise is exactly the identity channel
    # --------------------------------------------------------

    rho_ad0 = apply_amplitude_damping(
        rho_bell,
        gamma=0.0,
        qubit=0
    )

    rho_pd0 = apply_phase_damping(
        rho_bell,
        p=0.0,
        qubit=0
    )

    rho_dep0 = apply_depolarizing(
        rho_bell,
        p=0.0,
        qubit=0
    )

    assert np.allclose(
        rho_ad0,
        rho_bell,
        atol=1e-12
    )

    assert np.allclose(
        rho_pd0,
        rho_bell,
        atol=1e-12
    )

    assert np.allclose(
        rho_dep0,
        rho_bell,
        atol=1e-12
    )

    # --------------------------------------------------------
    # Test 3:
    # All channels preserve physical density matrices
    # --------------------------------------------------------

    for qubit in [0, 1]:

        for p in [
            0.0,
            0.20,
            0.50,
            1.0
        ]:

            rho_ad = apply_amplitude_damping(
                rho_bell,
                gamma=p,
                qubit=qubit
            )

            rho_pd = apply_phase_damping(
                rho_bell,
                p=p,
                qubit=qubit
            )

            rho_dep = apply_depolarizing(
                rho_bell,
                p=p,
                qubit=qubit
            )

            assert_density_matrix(
                rho_ad
            )

            assert_density_matrix(
                rho_pd
            )

            assert_density_matrix(
                rho_dep
            )

    # --------------------------------------------------------
    # Test 4:
    # Amplitude damping of one half of Bell state
    #
    # Exact result:
    #
    #     C = sqrt(1-gamma)
    # --------------------------------------------------------

    for gamma in [
        0.0,
        0.10,
        0.40,
        0.75,
        1.0
    ]:

        rho = apply_amplitude_damping(
            rho_bell,
            gamma=gamma,
            qubit=0
        )

        C_numeric = concurrence_mixed(
            rho
        )

        C_exact = np.sqrt(
            1.0 - gamma
        )

        assert np.isclose(
            C_numeric,
            C_exact,
            atol=1e-7
        ), (
            "Amplitude-damping concurrence failed: "
            f"{C_numeric} vs {C_exact}"
        )

    # --------------------------------------------------------
    # Test 5:
    # Pure dephasing of one half of Bell state
    #
    # With our convention:
    #
    #     C = 1-p
    # --------------------------------------------------------

    for p in [
        0.0,
        0.10,
        0.40,
        0.75,
        1.0
    ]:

        rho = apply_phase_damping(
            rho_bell,
            p=p,
            qubit=0
        )

        C_numeric = concurrence_mixed(
            rho
        )

        C_exact = (
            1.0 - p
        )

        assert np.isclose(
            C_numeric,
            C_exact,
            atol=1e-7
        ), (
            "Phase-damping concurrence failed: "
            f"{C_numeric} vs {C_exact}"
        )

    # --------------------------------------------------------
    # Test 6:
    # Single-qubit depolarizing noise on Bell state
    #
    # For our depolarizing convention:
    #
    #     C = max(0, 1 - 3p/2)
    # --------------------------------------------------------

    for p in [
        0.0,
        0.10,
        0.40,
        2.0 / 3.0,
        0.90,
        1.0
    ]:

        rho = apply_depolarizing(
            rho_bell,
            p=p,
            qubit=0
        )

        C_numeric = concurrence_mixed(
            rho
        )

        C_exact = max(
            0.0,
            1.0 - 1.5 * p
        )

        assert np.isclose(
            C_numeric,
            C_exact,
            atol=1e-7
        ), (
            "Depolarizing concurrence failed: "
            f"{C_numeric} vs {C_exact}"
        )

    # --------------------------------------------------------
    # Test 7:
    # Complete amplitude damping destroys Bell entanglement
    # --------------------------------------------------------

    rho_ad_full = apply_amplitude_damping(
        rho_bell,
        gamma=1.0,
        qubit=0
    )

    assert np.isclose(
        concurrence_mixed(rho_ad_full),
        0.0,
        atol=1e-12
    )

    # --------------------------------------------------------
    # Test 8:
    # Complete phase damping destroys Bell entanglement
    # --------------------------------------------------------

    rho_pd_full = apply_phase_damping(
        rho_bell,
        p=1.0,
        qubit=0
    )

    assert np.isclose(
        concurrence_mixed(rho_pd_full),
        0.0,
        atol=1e-12
    )

    # --------------------------------------------------------
    # Test 9:
    # Noise must reduce purity for a nontrivial Bell state
    # --------------------------------------------------------

    rho_test = apply_phase_damping(
        rho_bell,
        p=0.30,
        qubit=0
    )

    assert purity(
        rho_test
    ) < 1.0

    if verbose:

        print("Cell 2 unit tests: PASS")
        print(
            "  ✓ all Kraus maps satisfy Σ K†K = I"
        )
        print(
            "  ✓ zero-noise limits reproduce identity"
        )
        print(
            "  ✓ trace / Hermiticity / positivity preserved"
        )
        print(
            "  ✓ amplitude damping gives C = sqrt(1-γ)"
        )
        print(
            "  ✓ phase damping gives C = 1-p"
        )
        print(
            "  ✓ depolarizing gives C = max(0, 1-3p/2)"
        )
        print(
            "  ✓ complete damping/dephasing destroys Bell entanglement"
        )
        print(
            "  ✓ nontrivial noise produces mixed states"
        )


# Run tests immediately
run_cell2_unit_tests()

Cell 2 unit tests: PASS
  ✓ all Kraus maps satisfy Σ K†K = I
  ✓ zero-noise limits reproduce identity
  ✓ trace / Hermiticity / positivity preserved
  ✓ amplitude damping gives C = sqrt(1-γ)
  ✓ phase damping gives C = 1-p
  ✓ depolarizing gives C = max(0, 1-3p/2)
  ✓ complete damping/dephasing destroys Bell entanglement
  ✓ nontrivial noise produces mixed states


In [3]:
# ============================================================
# V2 — Cell 3
# Noisy Bell-state circuit with explicit noise locations
#
# Requires:
#   Cell 1 — density-matrix core
#   Cell 2 — validated noise channels
#
# New capability:
#
#   |00>
#     |
#   Ry(alpha)
#     |
#   [noise layer A]
#     |
#   CNOT
#     |
#   [noise layer B]
#     |
#   rho_out
#
# Each noise layer can independently contain:
#
#   amplitude damping: (q0, q1)
#   dephasing:         (q0, q1)
#   depolarizing:      (q0, q1)
#
# No optimizer yet.
# No finite-shot measurements yet.
# ============================================================


# ------------------------------------------------------------
# 1. Noise-layer configuration utilities
# ------------------------------------------------------------

def _noise_pair(value, name):
    """
    Convert a channel parameter into a two-qubit pair.

    Examples
    --------
    0.1
        -> (0.1, 0.1)

    (0.1, 0.0)
        -> q0 gets 0.1, q1 gets 0.0

    None
        -> (0.0, 0.0)
    """

    if value is None:
        return (0.0, 0.0)

    if np.isscalar(value):
        p = _check_probability(
            value,
            name
        )
        return (p, p)

    if len(value) != 2:
        raise ValueError(
            f"{name} must be a scalar or length-2 sequence."
        )

    return tuple(
        _check_probability(
            value[i],
            f"{name}[{i}]"
        )
        for i in range(2)
    )


def normalize_noise_layer(layer=None):
    """
    Return a complete normalized noise specification.

    Example
    -------

    {
        "amplitude":   (0.05, 0.02),
        "dephasing":   (0.10, 0.00),
        "depolarizing":(0.00, 0.01)
    }

    Missing entries default to zero.
    """

    if layer is None:
        layer = {}

    return {
        "amplitude": _noise_pair(
            layer.get(
                "amplitude",
                (0.0, 0.0)
            ),
            "amplitude"
        ),

        "dephasing": _noise_pair(
            layer.get(
                "dephasing",
                (0.0, 0.0)
            ),
            "dephasing"
        ),

        "depolarizing": _noise_pair(
            layer.get(
                "depolarizing",
                (0.0, 0.0)
            ),
            "depolarizing"
        ),
    }


# ------------------------------------------------------------
# 2. Apply one complete noise layer
#
# Fixed composition order:
#
#   amplitude damping
#       ->
#   dephasing
#       ->
#   depolarizing
#
# The order is explicit because general quantum channels
# need not commute.
# ------------------------------------------------------------

def apply_noise_layer(
    rho,
    layer=None
):
    """
    Apply a configurable local noise layer to both qubits.
    """

    config = normalize_noise_layer(
        layer
    )

    rho_out = np.array(
        rho,
        dtype=complex,
        copy=True
    )

    # --------------------------------------------------------
    # Amplitude damping
    # --------------------------------------------------------

    for qubit in (0, 1):

        gamma = config[
            "amplitude"
        ][qubit]

        if gamma > 0.0:

            rho_out = apply_amplitude_damping(
                rho_out,
                gamma=gamma,
                qubit=qubit
            )

    # --------------------------------------------------------
    # Pure dephasing
    # --------------------------------------------------------

    for qubit in (0, 1):

        p = config[
            "dephasing"
        ][qubit]

        if p > 0.0:

            rho_out = apply_phase_damping(
                rho_out,
                p=p,
                qubit=qubit
            )

    # --------------------------------------------------------
    # Depolarizing noise
    # --------------------------------------------------------

    for qubit in (0, 1):

        p = config[
            "depolarizing"
        ][qubit]

        if p > 0.0:

            rho_out = apply_depolarizing(
                rho_out,
                p=p,
                qubit=qubit
            )

    return rho_out


# ------------------------------------------------------------
# 3. Full noisy Bell-like circuit
# ------------------------------------------------------------

def prepare_noisy_density_matrix(
    alpha,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Prepare the Bell-like state with noise inserted at
    explicitly defined locations.

    Circuit
    -------

        |00>
          |
        Ry(alpha) on q0
          |
        noise_after_ry
          |
        CNOT
          |
        noise_after_cnot
          |
        rho_out
    """

    # Initial state
    rho = ket_to_rho(
        ket00
    )

    # --------------------------------------------------------
    # Ry(alpha) on first qubit
    # --------------------------------------------------------

    U_ry = np.kron(
        Ry(alpha),
        I2
    )

    rho = apply_unitary(
        rho,
        U_ry
    )

    # --------------------------------------------------------
    # Noise between Ry and CNOT
    # --------------------------------------------------------

    rho = apply_noise_layer(
        rho,
        noise_after_ry
    )

    # --------------------------------------------------------
    # CNOT
    # --------------------------------------------------------

    rho = apply_unitary(
        rho,
        CNOT
    )

    # --------------------------------------------------------
    # Noise after CNOT
    # --------------------------------------------------------

    rho = apply_noise_layer(
        rho,
        noise_after_cnot
    )

    return rho


# ------------------------------------------------------------
# 4. Circuit diagnostics
# ------------------------------------------------------------

def evaluate_noisy_circuit(
    alpha,
    target_alpha=None,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Run the noisy circuit and return useful exact-state metrics.

    This is still a simulator-level diagnostic:
    concurrence and fidelity are computed directly from rho.

    Finite-shot measurement comes later.
    """

    if target_alpha is None:
        target_alpha = alpha

    rho = prepare_noisy_density_matrix(
        alpha=alpha,
        noise_after_ry=noise_after_ry,
        noise_after_cnot=noise_after_cnot
    )

    psi_target = bell_like_ket(
        target_alpha
    )

    return {
        "rho": rho,

        "concurrence": concurrence_mixed(
            rho
        ),

        "fidelity": fidelity_with_pure_target(
            rho,
            psi_target
        ),

        "purity": purity(
            rho
        )
    }


# ------------------------------------------------------------
# 5. Cell 3 unit tests
# ------------------------------------------------------------

def run_cell3_unit_tests(verbose=True):

    alpha_bell = np.pi / 2

    psi_bell = bell_like_ket(
        alpha_bell
    )

    rho_bell = prepare_ideal_density_matrix(
        alpha_bell
    )

    # ========================================================
    # Test 1
    # Zero-noise noisy circuit reproduces Cell 1 exactly
    # ========================================================

    for alpha in [
        0.0,
        0.23,
        0.71,
        np.pi / 2,
        1.91
    ]:

        rho_noisy = prepare_noisy_density_matrix(
            alpha
        )

        rho_ideal = prepare_ideal_density_matrix(
            alpha
        )

        assert np.allclose(
            rho_noisy,
            rho_ideal,
            atol=1e-12
        ), (
            "Zero-noise circuit does not reproduce "
            f"ideal result at alpha={alpha}"
        )

    # ========================================================
    # Test 2
    # Post-CNOT amplitude damping:
    #
    # Bell state, one noisy qubit
    #
    #     C = sqrt(1-gamma)
    #
    # Fidelity:
    #
    #     F =
    #       [2 - gamma + 2 sqrt(1-gamma)] / 4
    # ========================================================

    for gamma in [
        0.0,
        0.10,
        0.40,
        0.75,
        1.0
    ]:

        rho = prepare_noisy_density_matrix(
            alpha_bell,
            noise_after_cnot={
                "amplitude": (
                    gamma,
                    0.0
                )
            }
        )

        C_numeric = concurrence_mixed(
            rho
        )

        C_exact = np.sqrt(
            1.0 - gamma
        )

        F_numeric = fidelity_with_pure_target(
            rho,
            psi_bell
        )

        F_exact = (
            2.0
            - gamma
            + 2.0 * np.sqrt(
                1.0 - gamma
            )
        ) / 4.0

        assert np.isclose(
            C_numeric,
            C_exact,
            atol=1e-7
        )

        assert np.isclose(
            F_numeric,
            F_exact,
            atol=1e-12
        )

    # ========================================================
    # Test 3
    # Post-CNOT pure dephasing:
    #
    #     C = 1-p
    #     F = 1-p/2
    # ========================================================

    for p in [
        0.0,
        0.10,
        0.40,
        0.75,
        1.0
    ]:

        rho = prepare_noisy_density_matrix(
            alpha_bell,
            noise_after_cnot={
                "dephasing": (
                    p,
                    0.0
                )
            }
        )

        C_numeric = concurrence_mixed(
            rho
        )

        C_exact = (
            1.0 - p
        )

        F_numeric = fidelity_with_pure_target(
            rho,
            psi_bell
        )

        F_exact = (
            1.0 - p / 2.0
        )

        assert np.isclose(
            C_numeric,
            C_exact,
            atol=1e-7
        )

        assert np.isclose(
            F_numeric,
            F_exact,
            atol=1e-12
        )

    # ========================================================
    # Test 4
    # Post-CNOT depolarizing noise:
    #
    #     C = max(0, 1 - 3p/2)
    #
    #     F = 1 - 3p/4
    # ========================================================

    for p in [
        0.0,
        0.10,
        0.40,
        2.0 / 3.0,
        1.0
    ]:

        rho = prepare_noisy_density_matrix(
            alpha_bell,
            noise_after_cnot={
                "depolarizing": (
                    p,
                    0.0
                )
            }
        )

        C_numeric = concurrence_mixed(
            rho
        )

        C_exact = max(
            0.0,
            1.0 - 1.5 * p
        )

        F_numeric = fidelity_with_pure_target(
            rho,
            psi_bell
        )

        F_exact = (
            1.0
            - 0.75 * p
        )

        assert np.isclose(
            C_numeric,
            C_exact,
            atol=1e-7
        )

        assert np.isclose(
            F_numeric,
            F_exact,
            atol=1e-12
        )

    # ========================================================
    # Test 5
    # Location matters:
    #
    # Before CNOT the second qubit is still |0>.
    #
    # Therefore:
    #
    #   amplitude damping on q1
    #
    # and
    #
    #   phase damping on q1
    #
    # before CNOT do nothing.
    # ========================================================

    for gamma in [
        0.20,
        0.80,
        1.0
    ]:

        rho = prepare_noisy_density_matrix(
            alpha_bell,
            noise_after_ry={
                "amplitude": (
                    0.0,
                    gamma
                )
            }
        )

        assert np.allclose(
            rho,
            rho_bell,
            atol=1e-12
        )

    for p in [
        0.20,
        0.80,
        1.0
    ]:

        rho = prepare_noisy_density_matrix(
            alpha_bell,
            noise_after_ry={
                "dephasing": (
                    0.0,
                    p
                )
            }
        )

        assert np.allclose(
            rho,
            rho_bell,
            atol=1e-12
        )

    # ========================================================
    # Test 6
    # Same noise applied AFTER entanglement formation
    # must have an effect.
    # ========================================================

    rho_after = prepare_noisy_density_matrix(
        alpha_bell,
        noise_after_cnot={
            "amplitude": (
                0.0,
                0.50
            )
        }
    )

    assert concurrence_mixed(
        rho_after
    ) < 1.0

    assert fidelity_with_pure_target(
        rho_after,
        psi_bell
    ) < 1.0

    # ========================================================
    # Test 7
    # Two-sided dephasing:
    #
    # coherence gets multiplied by
    #
    #     (1-p0)(1-p1)
    #
    # and therefore
    #
    #     C = (1-p0)(1-p1)
    # ========================================================

    test_pairs = [
        (0.10, 0.20),
        (0.30, 0.40),
        (0.50, 0.50),
        (1.00, 0.20)
    ]

    for p0, p1 in test_pairs:

        rho = prepare_noisy_density_matrix(
            alpha_bell,
            noise_after_cnot={
                "dephasing": (
                    p0,
                    p1
                )
            }
        )

        C_numeric = concurrence_mixed(
            rho
        )

        C_exact = (
            (1.0 - p0)
            * (1.0 - p1)
        )

        assert np.isclose(
            C_numeric,
            C_exact,
            atol=1e-7
        )

    # ========================================================
    # Test 8
    # General mixed noise remains a valid density matrix
    # ========================================================

    rho_mixed = prepare_noisy_density_matrix(
        alpha=0.83,

        noise_after_ry={
            "amplitude": (
                0.05,
                0.00
            ),
            "dephasing": (
                0.08,
                0.00
            )
        },

        noise_after_cnot={
            "amplitude": (
                0.07,
                0.04
            ),
            "dephasing": (
                0.03,
                0.06
            ),
            "depolarizing": (
                0.02,
                0.01
            )
        }
    )

    assert_density_matrix(
        rho_mixed
    )

    assert (
        0.0
        <= concurrence_mixed(rho_mixed)
        <= 1.0
    )

    assert (
        0.0
        <= purity(rho_mixed)
        <= 1.0 + 1e-12
    )

    # ========================================================
    # Report
    # ========================================================

    if verbose:

        print("Cell 3 unit tests: PASS")

        print(
            "  ✓ zero-noise circuit reproduces Cell 1"
        )

        print(
            "  ✓ post-CNOT amplitude damping matches analytic C and F"
        )

        print(
            "  ✓ post-CNOT dephasing matches analytic C and F"
        )

        print(
            "  ✓ post-CNOT depolarizing noise matches analytic C and F"
        )

        print(
            "  ✓ pre-CNOT noise on inactive target can be physically irrelevant"
        )

        print(
            "  ✓ same noise after entanglement formation degrades the state"
        )

        print(
            "  ✓ two-sided dephasing obeys C=(1-p0)(1-p1)"
        )

        print(
            "  ✓ combined noisy circuit remains a valid density matrix"
        )


# Run immediately
run_cell3_unit_tests()

Cell 3 unit tests: PASS
  ✓ zero-noise circuit reproduces Cell 1
  ✓ post-CNOT amplitude damping matches analytic C and F
  ✓ post-CNOT dephasing matches analytic C and F
  ✓ post-CNOT depolarizing noise matches analytic C and F
  ✓ pre-CNOT noise on inactive target can be physically irrelevant
  ✓ same noise after entanglement formation degrades the state
  ✓ two-sided dephasing obeys C=(1-p0)(1-p1)
  ✓ combined noisy circuit remains a valid density matrix


In [4]:
# ============================================================
# V2 — Cell 4
# Two-parameter Bell-like state:
# entanglement amplitude alpha + relative phase phi
#
# Requires:
#   Cell 1 — density-matrix core
#   Cell 2 — noise channels
#   Cell 3 — noisy circuit
#
# New idea:
#
#   |psi(alpha, phi)> =
#
#       cos(alpha/2)|00>
#       + exp(i phi) sin(alpha/2)|11>
#
# Concurrence:
#
#       C = |sin(alpha)|
#
# is independent of phi.
#
# Therefore concurrence alone does NOT determine
# the target state.
#
# No optimizer yet.
# No finite-shot measurement yet.
# ============================================================


# ------------------------------------------------------------
# 1. Phase gate
#
# P(phi) = diag(1, exp(i phi))
#
# Applied to q0 after the CNOT:
#
#   c|00> + s|11>
#
#       ->
#
#   c|00> + exp(i phi)s|11>
# ------------------------------------------------------------

def phase_gate(phi):
    """
    Single-qubit phase gate P(phi).
    """

    return np.array([
        [1.0, 0.0],
        [0.0, np.exp(1j * phi)]
    ], dtype=complex)


# ------------------------------------------------------------
# 2. Analytic two-parameter reference ket
# ------------------------------------------------------------

def bell_like_ket_phase(alpha, phi):
    """
    Analytic reference state:

        |psi(alpha,phi)> =
            cos(alpha/2)|00>
            + exp(i phi) sin(alpha/2)|11>
    """

    return np.array([
        np.cos(alpha / 2.0),
        0.0,
        0.0,
        np.exp(1j * phi) * np.sin(alpha / 2.0)
    ], dtype=complex)


# ------------------------------------------------------------
# 3. Ideal two-parameter density-matrix circuit
# ------------------------------------------------------------

def prepare_ideal_density_matrix_phase(
    alpha,
    phi
):
    """
    Prepare

        |psi(alpha,phi)>

    using the actual circuit:

        |00>
          |
        Ry(alpha) on q0
          |
        CNOT
          |
        P(phi) on q0
          |
        rho_out
    """

    rho = ket_to_rho(
        ket00
    )

    # Ry(alpha) on q0
    U_ry = np.kron(
        Ry(alpha),
        I2
    )

    rho = apply_unitary(
        rho,
        U_ry
    )

    # CNOT
    rho = apply_unitary(
        rho,
        CNOT
    )

    # Relative phase control on q0
    U_phi = np.kron(
        phase_gate(phi),
        I2
    )

    rho = apply_unitary(
        rho,
        U_phi
    )

    return rho


# ------------------------------------------------------------
# 4. Noisy two-parameter circuit
#
# Noise locations remain explicit:
#
#   |00>
#     |
#   Ry(alpha)
#     |
#   noise_after_ry
#     |
#   CNOT
#     |
#   P(phi)
#     |
#   noise_after_cnot
#     |
#   rho_out
#
# "noise_after_cnot" therefore means noise after the
# complete entangling + relative-phase control block.
# ------------------------------------------------------------

def prepare_noisy_density_matrix_phase(
    alpha,
    phi,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Two-parameter noisy Bell-state preparation.
    """

    rho = ket_to_rho(
        ket00
    )

    # --------------------------------------------------------
    # Ry(alpha)
    # --------------------------------------------------------

    U_ry = np.kron(
        Ry(alpha),
        I2
    )

    rho = apply_unitary(
        rho,
        U_ry
    )

    # --------------------------------------------------------
    # Noise before entanglement
    # --------------------------------------------------------

    rho = apply_noise_layer(
        rho,
        noise_after_ry
    )

    # --------------------------------------------------------
    # CNOT
    # --------------------------------------------------------

    rho = apply_unitary(
        rho,
        CNOT
    )

    # --------------------------------------------------------
    # Relative phase control
    # --------------------------------------------------------

    U_phi = np.kron(
        phase_gate(phi),
        I2
    )

    rho = apply_unitary(
        rho,
        U_phi
    )

    # --------------------------------------------------------
    # Final noise
    # --------------------------------------------------------

    rho = apply_noise_layer(
        rho,
        noise_after_cnot
    )

    return rho


# ------------------------------------------------------------
# 5. Exact analytic fidelity
#
# For
#
#   |psi_t> = |psi(alpha_t, phi_t)>
#
# and
#
#   |psi>   = |psi(alpha, phi)>
#
# overlap:
#
#   A =
#     cos(alpha_t/2) cos(alpha/2)
#     +
#     exp[i(phi-phi_t)]
#     sin(alpha_t/2) sin(alpha/2)
#
# Fidelity:
#
#   F = |A|^2
# ------------------------------------------------------------

def analytic_phase_state_fidelity(
    alpha,
    phi,
    target_alpha,
    target_phi
):
    """
    Exact pure-state fidelity for two Bell-like states.
    """

    amplitude = (
        np.cos(target_alpha / 2.0)
        * np.cos(alpha / 2.0)
        +
        np.exp(
            1j * (phi - target_phi)
        )
        * np.sin(target_alpha / 2.0)
        * np.sin(alpha / 2.0)
    )

    return float(
        np.abs(amplitude) ** 2
    )


# ------------------------------------------------------------
# 6. Circuit evaluator
# ------------------------------------------------------------

def evaluate_noisy_circuit_phase(
    alpha,
    phi,
    target_alpha=None,
    target_phi=0.0,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Exact-state diagnostics for the two-parameter circuit.
    """

    if target_alpha is None:
        target_alpha = alpha

    rho = prepare_noisy_density_matrix_phase(
        alpha=alpha,
        phi=phi,
        noise_after_ry=noise_after_ry,
        noise_after_cnot=noise_after_cnot
    )

    psi_target = bell_like_ket_phase(
        target_alpha,
        target_phi
    )

    return {
        "rho": rho,

        "concurrence": concurrence_mixed(
            rho
        ),

        "fidelity": fidelity_with_pure_target(
            rho,
            psi_target
        ),

        "purity": purity(
            rho
        )
    }


# ------------------------------------------------------------
# 7. Cell 4 unit tests
# ------------------------------------------------------------

def run_cell4_unit_tests(verbose=True):

    # ========================================================
    # Test 1
    # Circuit reproduces analytic complex state
    # ========================================================

    test_points = [
        (0.00,  0.00),
        (0.23,  0.41),
        (0.71, -0.83),
        (1.10,  2.20),
        (np.pi / 2, np.pi)
    ]

    for alpha, phi in test_points:

        rho_numeric = (
            prepare_ideal_density_matrix_phase(
                alpha,
                phi
            )
        )

        rho_exact = ket_to_rho(
            bell_like_ket_phase(
                alpha,
                phi
            )
        )

        assert_density_matrix(
            rho_numeric
        )

        assert np.allclose(
            rho_numeric,
            rho_exact,
            atol=1e-12
        ), (
            "Two-parameter circuit disagrees "
            f"with analytic state at "
            f"alpha={alpha}, phi={phi}"
        )

    # ========================================================
    # Test 2
    # phi = 0 reproduces Cell 1 / Cell 3 ideal circuit
    # ========================================================

    for alpha in [
        0.0,
        0.30,
        0.80,
        np.pi / 2
    ]:

        rho_phase = (
            prepare_ideal_density_matrix_phase(
                alpha,
                0.0
            )
        )

        rho_old = (
            prepare_ideal_density_matrix(
                alpha
            )
        )

        assert np.allclose(
            rho_phase,
            rho_old,
            atol=1e-12
        )

    # ========================================================
    # Test 3
    # Concurrence is independent of phi
    #
    #     C = |sin(alpha)|
    # ========================================================

    phase_values = [
        -2.7,
        -1.1,
        0.0,
        0.37,
        1.9,
        np.pi
    ]

    for alpha in [
        0.20,
        0.65,
        1.10,
        np.pi / 2
    ]:

        C_exact = abs(
            np.sin(alpha)
        )

        for phi in phase_values:

            rho = (
                prepare_ideal_density_matrix_phase(
                    alpha,
                    phi
                )
            )

            C_numeric = concurrence_mixed(
                rho
            )

            assert np.isclose(
                C_numeric,
                C_exact,
                atol=1e-7
            ), (
                "Concurrence incorrectly depends "
                f"on phase at alpha={alpha}, "
                f"phi={phi}"
            )

    # ========================================================
    # Test 4
    # General pure-state fidelity agrees with analytic formula
    # ========================================================

    fidelity_cases = [
        (
            0.40, 0.20,
            0.80, -0.30
        ),
        (
            0.90, 1.10,
            0.90, -0.70
        ),
        (
            np.pi / 2, 0.30,
            np.pi / 2, 1.20
        ),
        (
            0.25, -2.00,
            1.10, 2.40
        )
    ]

    for (
        alpha,
        phi,
        alpha_t,
        phi_t
    ) in fidelity_cases:

        rho = (
            prepare_ideal_density_matrix_phase(
                alpha,
                phi
            )
        )

        psi_target = bell_like_ket_phase(
            alpha_t,
            phi_t
        )

        F_numeric = (
            fidelity_with_pure_target(
                rho,
                psi_target
            )
        )

        F_exact = (
            analytic_phase_state_fidelity(
                alpha,
                phi,
                alpha_t,
                phi_t
            )
        )

        assert np.isclose(
            F_numeric,
            F_exact,
            atol=1e-12
        )

    # ========================================================
    # Test 5
    # Equal alpha:
    #
    #   F =
    #
    #   1 - sin^2(alpha)
    #       sin^2((phi-phi_t)/2)
    # ========================================================

    for alpha in [
        0.30,
        0.80,
        np.pi / 2
    ]:

        for delta_phi in [
            0.0,
            0.20,
            0.70,
            np.pi
        ]:

            phi_t = 0.31

            phi = (
                phi_t
                + delta_phi
            )

            rho = (
                prepare_ideal_density_matrix_phase(
                    alpha,
                    phi
                )
            )

            psi_target = (
                bell_like_ket_phase(
                    alpha,
                    phi_t
                )
            )

            F_numeric = (
                fidelity_with_pure_target(
                    rho,
                    psi_target
                )
            )

            F_exact = (
                1.0
                -
                np.sin(alpha) ** 2
                * np.sin(
                    delta_phi / 2.0
                ) ** 2
            )

            assert np.isclose(
                F_numeric,
                F_exact,
                atol=1e-12
            )

    # ========================================================
    # Test 6
    # Bell-state phase extremes
    #
    # phi = 0:
    #
    #   |Phi+>
    #
    # phi = pi:
    #
    #   |Phi->
    #
    # Both have concurrence 1,
    # but they are orthogonal.
    # ========================================================

    rho_plus = (
        prepare_ideal_density_matrix_phase(
            np.pi / 2,
            0.0
        )
    )

    rho_minus = (
        prepare_ideal_density_matrix_phase(
            np.pi / 2,
            np.pi
        )
    )

    psi_plus = bell_like_ket_phase(
        np.pi / 2,
        0.0
    )

    C_plus = concurrence_mixed(
        rho_plus
    )

    C_minus = concurrence_mixed(
        rho_minus
    )

    F_minus_to_plus = (
        fidelity_with_pure_target(
            rho_minus,
            psi_plus
        )
    )

    assert np.isclose(
        C_plus,
        1.0,
        atol=1e-8
    )

    assert np.isclose(
        C_minus,
        1.0,
        atol=1e-8
    )

    assert np.isclose(
        F_minus_to_plus,
        0.0,
        atol=1e-12
    )

    # ========================================================
    # Test 7
    # Phase periodicity:
    #
    # phi and phi + 2pi represent the same state.
    # ========================================================

    alpha = 0.83
    phi = 0.47

    rho_a = (
        prepare_ideal_density_matrix_phase(
            alpha,
            phi
        )
    )

    rho_b = (
        prepare_ideal_density_matrix_phase(
            alpha,
            phi + 2.0 * np.pi
        )
    )

    assert np.allclose(
        rho_a,
        rho_b,
        atol=1e-12
    )

    # ========================================================
    # Test 8
    # Noisy phase circuit remains a valid density matrix
    # ========================================================

    rho_noisy = (
        prepare_noisy_density_matrix_phase(
            alpha=0.91,
            phi=1.17,

            noise_after_ry={
                "amplitude": (
                    0.03,
                    0.00
                ),
                "dephasing": (
                    0.04,
                    0.00
                )
            },

            noise_after_cnot={
                "amplitude": (
                    0.06,
                    0.02
                ),
                "dephasing": (
                    0.05,
                    0.08
                ),
                "depolarizing": (
                    0.01,
                    0.02
                )
            }
        )
    )

    assert_density_matrix(
        rho_noisy
    )

    C_noisy = concurrence_mixed(
        rho_noisy
    )

    assert (
        0.0
        <= C_noisy
        <= 1.0
    )

    assert (
        0.0
        <= purity(rho_noisy)
        <= 1.0 + 1e-12
    )

    # ========================================================
    # Report
    # ========================================================

    if verbose:

        print("Cell 4 unit tests: PASS")

        print(
            "  ✓ circuit reproduces "
            "cos(alpha/2)|00> + exp(i phi)sin(alpha/2)|11>"
        )

        print(
            "  ✓ phi=0 reproduces the frozen V1/V2 baseline"
        )

        print(
            "  ✓ concurrence remains C=|sin(alpha)| for all phi"
        )

        print(
            "  ✓ general phase-state fidelity matches analytic result"
        )

        print(
            "  ✓ equal-alpha fidelity obeys "
            "1 - sin²(alpha) sin²(Δphi/2)"
        )

        print(
            "  ✓ |Phi+> and |Phi-> both have C=1 but F=0"
        )

        print(
            "  ✓ phase is 2π-periodic"
        )

        print(
            "  ✓ noisy two-parameter circuit remains physical"
        )


# Run immediately
run_cell4_unit_tests()

Cell 4 unit tests: PASS
  ✓ circuit reproduces cos(alpha/2)|00> + exp(i phi)sin(alpha/2)|11>
  ✓ phi=0 reproduces the frozen V1/V2 baseline
  ✓ concurrence remains C=|sin(alpha)| for all phi
  ✓ general phase-state fidelity matches analytic result
  ✓ equal-alpha fidelity obeys 1 - sin²(alpha) sin²(Δphi/2)
  ✓ |Phi+> and |Phi-> both have C=1 but F=0
  ✓ phase is 2π-periodic
  ✓ noisy two-parameter circuit remains physical


In [5]:
# ============================================================
# V2 — Cell 5
# Hybrid optimization:
#
#   A. concurrence-only control
#   B. state-aware concurrence + fidelity control
#
# Requires Cells 1–4.
#
# Key result:
#
#   concurrence-only:
#       alpha is controllable
#       phi is unidentifiable
#
#   state-aware:
#       alpha and phi are both controllable
#
# No finite-shot measurement yet.
# The optimizer still receives exact simulator metrics.
# ============================================================

from scipy.optimize import minimize, minimize_scalar


# ------------------------------------------------------------
# 1. Phase utilities
# ------------------------------------------------------------

def wrap_phase(phi):
    """
    Map phase to [-pi, pi).
    """
    return (
        (float(phi) + np.pi)
        % (2.0 * np.pi)
        - np.pi
    )


def phase_distance(phi_a, phi_b):
    """
    Smallest signed angular difference phi_a - phi_b.
    """
    return wrap_phase(
        phi_a - phi_b
    )


# ------------------------------------------------------------
# 2. Target construction
#
# Same principal-alpha convention as V1:
#
#     alpha_target = arcsin(C_target)
#
# but now the target also has a relative phase.
# ------------------------------------------------------------

def target_from_concurrence_phase(
    target_concurrence,
    target_phi=0.0
):
    """
    Construct a fully specified Bell-like target state.
    """

    C_target = float(
        target_concurrence
    )

    if not (
        0.0 <= C_target <= 1.0
    ):
        raise ValueError(
            "target_concurrence must lie in [0, 1]."
        )

    alpha_target = np.arcsin(
        C_target
    )

    phi_target = wrap_phase(
        target_phi
    )

    psi_target = bell_like_ket_phase(
        alpha_target,
        phi_target
    )

    return (
        alpha_target,
        phi_target,
        psi_target
    )


# ------------------------------------------------------------
# 3. Exact simulator evaluation
#
# This is still an exact-state calculation.
# Cell 6 will replace direct access to rho-derived metrics
# with finite-shot classical measurement data.
# ------------------------------------------------------------

def evaluate_v2_state(
    alpha,
    phi,
    target_concurrence,
    target_phi=0.0,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Evaluate the exact noisy V2 circuit against a
    fully specified target.
    """

    (
        alpha_target,
        phi_target,
        psi_target
    ) = target_from_concurrence_phase(
        target_concurrence,
        target_phi
    )

    rho = prepare_noisy_density_matrix_phase(
        alpha=alpha,
        phi=phi,
        noise_after_ry=noise_after_ry,
        noise_after_cnot=noise_after_cnot
    )

    C = concurrence_mixed(
        rho
    )

    F = fidelity_with_pure_target(
        rho,
        psi_target
    )

    return {
        "rho": rho,

        "alpha": float(alpha),
        "phi": wrap_phase(phi),

        "target_alpha": alpha_target,
        "target_phi": phi_target,

        "target_concurrence": float(
            target_concurrence
        ),

        "concurrence": C,

        "concurrence_error": abs(
            C - target_concurrence
        ),

        "fidelity": F,

        "infidelity": 1.0 - F,

        "purity": purity(
            rho
        ),

        "phase_error": phase_distance(
            phi,
            phi_target
        )
    }


# ------------------------------------------------------------
# 4. Concurrence-only objective
#
# IMPORTANT:
#
# phi is deliberately NOT optimized.
#
# Since
#
#     C(alpha,phi) = |sin(alpha)|
#
# in the ideal circuit, concurrence provides no
# information about phi.
#
# Letting a numerical optimizer move phi would only allow
# arbitrary motion along a flat direction.
# ------------------------------------------------------------

def run_concurrence_only_optimizer(
    target_concurrence,
    alpha_init=0.1,
    phi_init=0.0,
    noise_after_ry=None,
    noise_after_cnot=None,
    maxiter=200
):
    """
    Hybrid optimizer using concurrence only.

    alpha is optimized.
    phi remains uncontrolled at phi_init.
    """

    (
        alpha_target,
        phi_target,
        psi_target
    ) = target_from_concurrence_phase(
        target_concurrence,
        target_phi=0.0
    )

    phi_fixed = wrap_phase(
        phi_init
    )

    history = []

    def objective(alpha):

        result = evaluate_v2_state(
            alpha=float(alpha),
            phi=phi_fixed,
            target_concurrence=target_concurrence,
            target_phi=phi_target,
            noise_after_ry=noise_after_ry,
            noise_after_cnot=noise_after_cnot
        )

        loss = (
            result["concurrence_error"] ** 2
        )

        history.append({
            "alpha": float(alpha),
            "phi": phi_fixed,
            "loss": float(loss),
            "concurrence": result["concurrence"],
            "fidelity": result["fidelity"]
        })

        return loss

    opt = minimize_scalar(
        objective,
        bounds=(
            0.0,
            np.pi / 2.0
        ),
        method="bounded",
        options={
            "xatol": 1e-12,
            "maxiter": int(maxiter)
        }
    )

    alpha_opt = float(
        opt.x
    )

    final = evaluate_v2_state(
        alpha=alpha_opt,
        phi=phi_fixed,
        target_concurrence=target_concurrence,
        target_phi=phi_target,
        noise_after_ry=noise_after_ry,
        noise_after_cnot=noise_after_cnot
    )

    final.update({
        "objective": "concurrence_only",

        "alpha_init": float(
            alpha_init
        ),

        "phi_init": phi_fixed,

        "alpha_opt": alpha_opt,

        "phi_opt": phi_fixed,

        "phi_identifiable": False,

        "loss": (
            final["concurrence_error"] ** 2
        ),

        "optimizer_success": bool(
            opt.success
        ),

        "optimizer_message": str(
            opt.message
        ),

        "nfev": int(
            opt.nfev
        ),

        "history": history
    })

    return final


# ------------------------------------------------------------
# 5. State-aware objective
#
# Now both alpha and phi are optimized.
#
# Loss:
#
#   L =
#       w_C (C - C_target)^2
#       +
#       w_F (1 - F)
#
# Fidelity supplies the missing phase information.
# ------------------------------------------------------------

def run_state_aware_optimizer(
    target_concurrence,
    target_phi=0.0,
    alpha_init=0.1,
    phi_init=0.0,
    weight_concurrence=1.0,
    weight_fidelity=1.0,
    noise_after_ry=None,
    noise_after_cnot=None,
    maxiter=500
):
    """
    Hybrid optimizer using concurrence + target-state fidelity.
    """

    weight_concurrence = float(
        weight_concurrence
    )

    weight_fidelity = float(
        weight_fidelity
    )

    if weight_concurrence < 0.0:
        raise ValueError(
            "weight_concurrence must be non-negative."
        )

    if weight_fidelity <= 0.0:
        raise ValueError(
            "weight_fidelity must be positive."
        )

    (
        alpha_target,
        phi_target,
        psi_target
    ) = target_from_concurrence_phase(
        target_concurrence,
        target_phi
    )

    history = []

    def objective(x):

        alpha = float(
            x[0]
        )

        phi = float(
            x[1]
        )

        result = evaluate_v2_state(
            alpha=alpha,
            phi=phi,
            target_concurrence=target_concurrence,
            target_phi=phi_target,
            noise_after_ry=noise_after_ry,
            noise_after_cnot=noise_after_cnot
        )

        loss_C = (
            result["concurrence_error"] ** 2
        )

        loss_F = (
            1.0 - result["fidelity"]
        )

        loss = (
            weight_concurrence * loss_C
            +
            weight_fidelity * loss_F
        )

        history.append({
            "alpha": alpha,
            "phi": wrap_phase(phi),
            "loss": float(loss),
            "loss_C": float(loss_C),
            "loss_F": float(loss_F),
            "concurrence": result["concurrence"],
            "fidelity": result["fidelity"]
        })

        return float(
            loss
        )

    x0 = np.array([
        np.clip(
            float(alpha_init),
            0.0,
            np.pi / 2.0
        ),
        wrap_phase(
            phi_init
        )
    ])

    opt = minimize(
        objective,
        x0=x0,
        method="Powell",
        bounds=[
            (
                0.0,
                np.pi / 2.0
            ),
            (
                -np.pi,
                np.pi
            )
        ],
        options={
            "xtol": 1e-10,
            "ftol": 1e-12,
            "maxiter": int(maxiter)
        }
    )

    alpha_opt = float(
        opt.x[0]
    )

    phi_opt = wrap_phase(
        opt.x[1]
    )

    final = evaluate_v2_state(
        alpha=alpha_opt,
        phi=phi_opt,
        target_concurrence=target_concurrence,
        target_phi=phi_target,
        noise_after_ry=noise_after_ry,
        noise_after_cnot=noise_after_cnot
    )

    loss_C = (
        final["concurrence_error"] ** 2
    )

    loss_F = (
        1.0 - final["fidelity"]
    )

    final.update({
        "objective": "state_aware",

        "alpha_init": float(
            alpha_init
        ),

        "phi_init": wrap_phase(
            phi_init
        ),

        "alpha_opt": alpha_opt,

        "phi_opt": phi_opt,

        "phi_identifiable": True,

        "loss": (
            weight_concurrence * loss_C
            +
            weight_fidelity * loss_F
        ),

        "loss_C": loss_C,

        "loss_F": loss_F,

        "optimizer_success": bool(
            opt.success
        ),

        "optimizer_message": str(
            opt.message
        ),

        "nfev": int(
            opt.nfev
        ),

        "history": history
    })

    return final


# ------------------------------------------------------------
# 6. Cell 5 unit tests
# ------------------------------------------------------------

def run_cell5_unit_tests(verbose=True):

    # Common test target
    C_target = 0.80
    phi_target = 0.40

    alpha_target = np.arcsin(
        C_target
    )

    # Deliberately incorrect phase
    phi_wrong = 2.40

    # ========================================================
    # Test 1
    # Concurrence objective is flat in phi
    # ========================================================

    phase_samples = [
        -2.8,
        -1.0,
        0.0,
        0.7,
        2.4
    ]

    C_values = []

    for phi in phase_samples:

        rho = prepare_ideal_density_matrix_phase(
            alpha_target,
            phi
        )

        C_values.append(
            concurrence_mixed(
                rho
            )
        )

    assert np.allclose(
        C_values,
        C_target,
        atol=1e-7
    )

    # ========================================================
    # Test 2
    # Fidelity is NOT flat in phi
    # ========================================================

    psi_target = bell_like_ket_phase(
        alpha_target,
        phi_target
    )

    rho_right_phase = (
        prepare_ideal_density_matrix_phase(
            alpha_target,
            phi_target
        )
    )

    rho_wrong_phase = (
        prepare_ideal_density_matrix_phase(
            alpha_target,
            phi_wrong
        )
    )

    F_right = fidelity_with_pure_target(
        rho_right_phase,
        psi_target
    )

    F_wrong = fidelity_with_pure_target(
        rho_wrong_phase,
        psi_target
    )

    assert np.isclose(
        F_right,
        1.0,
        atol=1e-12
    )

    assert F_wrong < 0.70

    # ========================================================
    # Test 3
    # Concurrence-only optimizer succeeds at C
    # but cannot fix the wrong phase.
    #
    # For this test its target phase is zero by construction.
    # ========================================================

    result_C = (
        run_concurrence_only_optimizer(
            target_concurrence=C_target,
            alpha_init=0.10,
            phi_init=phi_wrong,
            maxiter=300
        )
    )

    assert (
        result_C["concurrence_error"]
        < 1e-6
    )

    assert np.isclose(
        result_C["phi_opt"],
        wrap_phase(phi_wrong),
        atol=1e-12
    )

    assert (
        result_C["phi_identifiable"]
        is False
    )

    # It has matched entanglement,
    # but not the intended phi=0 target state.
    assert (
        result_C["fidelity"]
        < 0.70
    )

    # ========================================================
    # Test 4
    # State-aware optimizer recovers both alpha and phi
    # ========================================================

    result_S = (
        run_state_aware_optimizer(
            target_concurrence=C_target,
            target_phi=phi_target,
            alpha_init=0.20,
            phi_init=phi_wrong,
            weight_concurrence=1.0,
            weight_fidelity=1.0,
            maxiter=500
        )
    )

    assert (
        result_S["optimizer_success"]
        is True
    )

    assert (
        result_S["concurrence_error"]
        < 1e-6
    )

    assert (
        result_S["fidelity"]
        > 1.0 - 1e-10
    )

    assert abs(
        phase_distance(
            result_S["phi_opt"],
            phi_target
        )
    ) < 1e-5

    assert np.isclose(
        result_S["alpha_opt"],
        alpha_target,
        atol=1e-5
    )

    assert (
        result_S["phi_identifiable"]
        is True
    )

    # ========================================================
    # Test 5
    # If the uncontrolled phase already happens to be right,
    # concurrence-only optimization recovers the correct state.
    # ========================================================

    result_C_good_phase = (
        run_concurrence_only_optimizer(
            target_concurrence=C_target,
            alpha_init=0.10,
            phi_init=0.0,
            maxiter=300
        )
    )

    assert (
        result_C_good_phase[
            "concurrence_error"
        ]
        < 1e-6
    )

    assert (
        result_C_good_phase[
            "fidelity"
        ]
        > 1.0 - 1e-10
    )

    # ========================================================
    # Test 6
    # Exact target gives zero state-aware objective
    # ========================================================

    exact = evaluate_v2_state(
        alpha=alpha_target,
        phi=phi_target,
        target_concurrence=C_target,
        target_phi=phi_target
    )

    exact_loss = (
        exact["concurrence_error"] ** 2
        +
        exact["infidelity"]
    )

    assert np.isclose(
        exact_loss,
        0.0,
        atol=1e-12
    )

    # ========================================================
    # Test 7
    # Optimized output remains a physical density matrix
    # ========================================================

    assert_density_matrix(
        result_S["rho"]
    )

    assert (
        0.0
        <= result_S["purity"]
        <= 1.0 + 1e-12
    )

    # ========================================================
    # Report
    # ========================================================

    if verbose:

        print("Cell 5 unit tests: PASS")

        print(
            "  ✓ concurrence objective is flat in phi"
        )

        print(
            "  ✓ fidelity resolves the missing phase information"
        )

        print(
            "  ✓ concurrence-only control reaches C_target but can miss the target state"
        )

        print(
            "  ✓ state-aware control recovers both alpha and phi"
        )

        print(
            "  ✓ concurrence-only control succeeds when the uncontrolled phase is already correct"
        )

        print(
            "  ✓ exact target gives zero state-aware loss"
        )

        print(
            "  ✓ optimized output remains a physical density matrix"
        )


# Run immediately
run_cell5_unit_tests()

Cell 5 unit tests: PASS
  ✓ concurrence objective is flat in phi
  ✓ fidelity resolves the missing phase information
  ✓ concurrence-only control reaches C_target but can miss the target state
  ✓ state-aware control recovers both alpha and phi
  ✓ concurrence-only control succeeds when the uncontrolled phase is already correct
  ✓ exact target gives zero state-aware loss
  ✓ optimized output remains a physical density matrix


In [6]:
# ============================================================
# V2 — Cell 6
# Finite-shot Pauli measurements and state reconstruction
#
# Requires Cells 1–5.
#
# New capability:
#
#   quantum state rho
#       ->
#   finite-shot Pauli measurements
#       ->
#   classical expectation estimates
#       ->
#   reconstructed rho_hat
#       ->
#   estimated concurrence and fidelity
#
# No optimizer yet.
#
# Notes:
#   - 15 nontrivial two-qubit Pauli observables are sampled.
#   - Sampling is performed from the exact +/-1 outcome
#     probabilities of each Pauli observable.
#   - Linear inversion can produce a slightly nonphysical
#     density matrix at finite shots.
#   - A simple PSD projection is therefore also provided.
# ============================================================


# ------------------------------------------------------------
# 1. Pauli basis
# ------------------------------------------------------------

PAULI = {
    "I": I2,
    "X": X,
    "Y": Y,
    "Z": Z
}

PAULI_LABELS = [
    a + b
    for a in "IXYZ"
    for b in "IXYZ"
    if a + b != "II"
]


def pauli_operator(label):
    """
    Return the two-qubit Pauli operator associated with label.

    Examples:
        "XI" -> X ⊗ I
        "YZ" -> Y ⊗ Z
    """

    if (
        not isinstance(label, str)
        or len(label) != 2
        or label[0] not in PAULI
        or label[1] not in PAULI
    ):
        raise ValueError(
            "label must be a two-character Pauli string "
            "using I, X, Y, Z."
        )

    return np.kron(
        PAULI[label[0]],
        PAULI[label[1]]
    )


# ------------------------------------------------------------
# 2. Exact Pauli expectation value
# ------------------------------------------------------------

def pauli_expectation(
    rho,
    label
):
    """
    Compute

        <P> = Tr(rho P)

    for a two-qubit Pauli observable P.
    """

    rho = np.asarray(
        rho,
        dtype=complex
    )

    if rho.shape != (4, 4):
        raise ValueError(
            "rho must be a 4x4 two-qubit density matrix."
        )

    P = pauli_operator(
        label
    )

    value = np.trace(
        rho @ P
    )

    value = float(
        np.real_if_close(
            value
        ).real
    )

    # Remove tiny floating-point excursions.
    return float(
        np.clip(
            value,
            -1.0,
            1.0
        )
    )


# ------------------------------------------------------------
# 3. Finite-shot Pauli measurement
#
# A Pauli observable has eigenvalues +/-1.
#
# If
#
#     m = <P>
#
# then
#
#     p(+) = (1+m)/2
#     p(-) = (1-m)/2
#
# Sampling the binomial distribution is equivalent to
# performing independent projective measurements of P.
# ------------------------------------------------------------

def sample_pauli_expectation(
    rho,
    label,
    shots,
    rng
):
    """
    Estimate a Pauli expectation value from finite shots.

    Returns a dictionary containing the exact expectation,
    sampled expectation, and +/- outcome counts.
    """

    shots = int(
        shots
    )

    if shots <= 0:
        raise ValueError(
            "shots must be a positive integer."
        )

    exact = pauli_expectation(
        rho,
        label
    )

    p_plus = (
        1.0 + exact
    ) / 2.0

    p_plus = float(
        np.clip(
            p_plus,
            0.0,
            1.0
        )
    )

    n_plus = int(
        rng.binomial(
            shots,
            p_plus
        )
    )

    n_minus = (
        shots - n_plus
    )

    estimate = (
        n_plus - n_minus
    ) / shots

    return {
        "label": label,
        "shots": shots,

        "exact": exact,
        "estimate": float(
            estimate
        ),

        "n_plus": n_plus,
        "n_minus": n_minus
    }


# ------------------------------------------------------------
# 4. Exact Pauli table
# ------------------------------------------------------------

def exact_pauli_table(rho):
    """
    Return all 15 nontrivial Pauli expectation values.
    """

    return {
        label: pauli_expectation(
            rho,
            label
        )
        for label in PAULI_LABELS
    }


# ------------------------------------------------------------
# 5. Finite-shot Pauli table
# ------------------------------------------------------------

def sample_pauli_table(
    rho,
    shots_per_observable=2000,
    seed=0
):
    """
    Sample every nontrivial two-qubit Pauli observable.
    """

    shots_per_observable = int(
        shots_per_observable
    )

    if shots_per_observable <= 0:
        raise ValueError(
            "shots_per_observable must be positive."
        )

    rng = np.random.default_rng(
        seed
    )

    records = {}

    for label in PAULI_LABELS:

        records[label] = (
            sample_pauli_expectation(
                rho=rho,
                label=label,
                shots=shots_per_observable,
                rng=rng
            )
        )

    return records


# ------------------------------------------------------------
# 6. Reconstruct rho from Pauli expectations
#
# For two qubits:
#
#   rho =
#
#     1/4 sum_{mu,nu}
#
#       <sigma_mu ⊗ sigma_nu>
#
#       sigma_mu ⊗ sigma_nu
#
# with
#
#     <II> = 1.
# ------------------------------------------------------------

def rho_from_pauli_expectations(
    expectations
):
    """
    Linear-inversion reconstruction of a two-qubit density
    matrix from all 15 nontrivial Pauli expectations.
    """

    missing = [
        label
        for label in PAULI_LABELS
        if label not in expectations
    ]

    if missing:
        raise ValueError(
            "Missing Pauli expectations: "
            + ", ".join(missing)
        )

    rho = pauli_operator(
        "II"
    ).astype(
        complex
    )

    for label in PAULI_LABELS:

        rho += (
            float(
                expectations[label]
            )
            * pauli_operator(label)
        )

    rho /= 4.0

    # Explicitly enforce Hermiticity against
    # floating-point roundoff.
    rho = (
        rho
        + rho.conj().T
    ) / 2.0

    return rho


# ------------------------------------------------------------
# 7. Project finite-shot estimate onto physical state space
#
# Linear tomography can produce tiny negative eigenvalues.
#
# Here:
#
#   1. Hermitize
#   2. diagonalize
#   3. clip negative eigenvalues
#   4. renormalize trace
#
# This is a simple physicalization step, NOT full
# maximum-likelihood tomography.
# ------------------------------------------------------------

def project_to_physical_density_matrix(
    rho
):
    """
    Project a Hermitian trace-one estimate onto a positive
    semidefinite trace-one density matrix.
    """

    rho = np.asarray(
        rho,
        dtype=complex
    )

    rho_h = (
        rho
        + rho.conj().T
    ) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(
        rho_h
    )

    eigenvalues = np.clip(
        eigenvalues,
        0.0,
        None
    )

    total = np.sum(
        eigenvalues
    )

    if total <= 1e-15:

        return (
            np.eye(
                4,
                dtype=complex
            )
            / 4.0
        )

    eigenvalues /= total

    rho_physical = (
        eigenvectors
        @ np.diag(eigenvalues)
        @ eigenvectors.conj().T
    )

    rho_physical = (
        rho_physical
        + rho_physical.conj().T
    ) / 2.0

    return rho_physical


# ------------------------------------------------------------
# 8. Full finite-shot tomography pipeline
# ------------------------------------------------------------

def run_pauli_tomography(
    rho,
    shots_per_observable=2000,
    seed=0
):
    """
    Perform finite-shot Pauli tomography.

    Returns:
        measurement records
        expectation estimates
        raw linear-inversion rho
        physicalized rho
    """

    records = sample_pauli_table(
        rho=rho,
        shots_per_observable=shots_per_observable,
        seed=seed
    )

    estimates = {
        label: records[label]["estimate"]
        for label in PAULI_LABELS
    }

    rho_raw = rho_from_pauli_expectations(
        estimates
    )

    rho_physical = (
        project_to_physical_density_matrix(
            rho_raw
        )
    )

    return {
        "records": records,
        "expectations": estimates,
        "rho_raw": rho_raw,
        "rho_physical": rho_physical,
        "shots_per_observable": int(
            shots_per_observable
        ),
        "total_observable_shots": (
            len(PAULI_LABELS)
            * int(shots_per_observable)
        ),
        "seed": int(seed)
    }


# ------------------------------------------------------------
# 9. Measurement-derived state metrics
# ------------------------------------------------------------

def evaluate_tomographic_state(
    rho_true,
    psi_target,
    shots_per_observable=2000,
    seed=0
):
    """
    Compare exact-state metrics with finite-shot
    tomographic estimates.
    """

    tomography = run_pauli_tomography(
        rho=rho_true,
        shots_per_observable=shots_per_observable,
        seed=seed
    )

    rho_est = tomography[
        "rho_physical"
    ]

    return {
        "tomography": tomography,

        "exact_concurrence": concurrence_mixed(
            rho_true
        ),

        "estimated_concurrence": concurrence_mixed(
            rho_est
        ),

        "exact_fidelity": fidelity_with_pure_target(
            rho_true,
            psi_target
        ),

        "estimated_fidelity": fidelity_with_pure_target(
            rho_est,
            psi_target
        ),

        "exact_purity": purity(
            rho_true
        ),

        "estimated_purity": purity(
            rho_est
        )
    }


# ------------------------------------------------------------
# 10. Cell 6 unit tests
# ------------------------------------------------------------

def run_cell6_unit_tests(verbose=True):

    # ========================================================
    # Test state:
    # |Phi+>
    # ========================================================

    alpha_bell = (
        np.pi / 2
    )

    phi_bell = 0.0

    rho_bell = (
        prepare_ideal_density_matrix_phase(
            alpha_bell,
            phi_bell
        )
    )

    psi_bell = (
        bell_like_ket_phase(
            alpha_bell,
            phi_bell
        )
    )

    # ========================================================
    # Test 1
    # Known Bell stabilizers
    #
    # |Phi+>:
    #
    #   <XX> = +1
    #   <YY> = -1
    #   <ZZ> = +1
    #
    # local Pauli expectations vanish.
    # ========================================================

    assert np.isclose(
        pauli_expectation(
            rho_bell,
            "XX"
        ),
        1.0,
        atol=1e-12
    )

    assert np.isclose(
        pauli_expectation(
            rho_bell,
            "YY"
        ),
        -1.0,
        atol=1e-12
    )

    assert np.isclose(
        pauli_expectation(
            rho_bell,
            "ZZ"
        ),
        1.0,
        atol=1e-12
    )

    for label in [
        "XI",
        "YI",
        "ZI",
        "IX",
        "IY",
        "IZ"
    ]:

        assert np.isclose(
            pauli_expectation(
                rho_bell,
                label
            ),
            0.0,
            atol=1e-12
        )

    # ========================================================
    # Test 2
    # |00> gives expected Z observables
    # ========================================================

    rho_00 = ket_to_rho(
        ket00
    )

    assert np.isclose(
        pauli_expectation(
            rho_00,
            "ZI"
        ),
        1.0,
        atol=1e-12
    )

    assert np.isclose(
        pauli_expectation(
            rho_00,
            "IZ"
        ),
        1.0,
        atol=1e-12
    )

    assert np.isclose(
        pauli_expectation(
            rho_00,
            "ZZ"
        ),
        1.0,
        atol=1e-12
    )

    assert np.isclose(
        pauli_expectation(
            rho_00,
            "XX"
        ),
        0.0,
        atol=1e-12
    )

    # ========================================================
    # Test 3
    # Exact Pauli reconstruction recovers an arbitrary
    # known two-parameter state.
    # ========================================================

    rho_test = (
        prepare_ideal_density_matrix_phase(
            alpha=0.83,
            phi=1.17
        )
    )

    exact_table = exact_pauli_table(
        rho_test
    )

    rho_reconstructed = (
        rho_from_pauli_expectations(
            exact_table
        )
    )

    assert np.allclose(
        rho_reconstructed,
        rho_test,
        atol=1e-12
    )

    # ========================================================
    # Test 4
    # Exact Pauli reconstruction also recovers a noisy
    # mixed state.
    # ========================================================

    rho_noisy = (
        prepare_noisy_density_matrix_phase(
            alpha=0.91,
            phi=0.73,

            noise_after_ry={
                "amplitude": (
                    0.04,
                    0.00
                ),
                "dephasing": (
                    0.03,
                    0.00
                )
            },

            noise_after_cnot={
                "amplitude": (
                    0.07,
                    0.05
                ),
                "dephasing": (
                    0.04,
                    0.06
                ),
                "depolarizing": (
                    0.02,
                    0.01
                )
            }
        )
    )

    exact_noisy_table = (
        exact_pauli_table(
            rho_noisy
        )
    )

    rho_noisy_reconstructed = (
        rho_from_pauli_expectations(
            exact_noisy_table
        )
    )

    assert np.allclose(
        rho_noisy_reconstructed,
        rho_noisy,
        atol=1e-12
    )

    # ========================================================
    # Test 5
    # Finite-shot sampling is deterministic under fixed seed
    # ========================================================

    records_a = sample_pauli_table(
        rho_bell,
        shots_per_observable=1000,
        seed=1234
    )

    records_b = sample_pauli_table(
        rho_bell,
        shots_per_observable=1000,
        seed=1234
    )

    for label in PAULI_LABELS:

        assert (
            records_a[label]["n_plus"]
            ==
            records_b[label]["n_plus"]
        )

        assert (
            records_a[label]["n_minus"]
            ==
            records_b[label]["n_minus"]
        )

    # ========================================================
    # Test 6
    # Finite-shot Bell estimates converge near exact values
    #
    # Fixed seed makes this regression test reproducible.
    # ========================================================

    shots = 5000

    sampled = sample_pauli_table(
        rho_bell,
        shots_per_observable=shots,
        seed=12345
    )

    exact = exact_pauli_table(
        rho_bell
    )

    max_error = max(
        abs(
            sampled[label]["estimate"]
            -
            exact[label]
        )
        for label in PAULI_LABELS
    )

    assert (
        max_error < 0.05
    ), (
        "Finite-shot Pauli estimates deviated "
        f"too strongly: max error = {max_error}"
    )

    # ========================================================
    # Test 7
    # Finite-shot tomography produces a physical estimate
    # after PSD projection
    # ========================================================

    tomo = run_pauli_tomography(
        rho_bell,
        shots_per_observable=5000,
        seed=12345
    )

    rho_est = tomo[
        "rho_physical"
    ]

    assert_density_matrix(
        rho_est,
        atol=1e-10
    )

    # ========================================================
    # Test 8
    # Bell-state fidelity is recovered accurately
    # ========================================================

    F_est = fidelity_with_pure_target(
        rho_est,
        psi_bell
    )

    assert (
        F_est > 0.99
    ), (
        f"Finite-shot Bell fidelity too low: {F_est}"
    )

    # ========================================================
    # Test 9
    # Bell-state concurrence is recovered accurately
    # ========================================================

    C_est = concurrence_mixed(
        rho_est
    )

    assert (
        C_est > 0.98
    ), (
        f"Finite-shot Bell concurrence too low: {C_est}"
    )

    # ========================================================
    # Test 10
    # Tomographic metric wrapper is internally consistent
    # ========================================================

    result = evaluate_tomographic_state(
        rho_true=rho_bell,
        psi_target=psi_bell,
        shots_per_observable=5000,
        seed=12345
    )

    assert np.isclose(
        result["exact_concurrence"],
        1.0,
        atol=1e-8
    )

    assert np.isclose(
        result["exact_fidelity"],
        1.0,
        atol=1e-12
    )

    assert (
        result["estimated_concurrence"]
        > 0.98
    )

    assert (
        result["estimated_fidelity"]
        > 0.99
    )

    # ========================================================
    # Test 11
    # Invalid shot count is rejected
    # ========================================================

    try:

        sample_pauli_table(
            rho_bell,
            shots_per_observable=0,
            seed=0
        )

        raise AssertionError(
            "shots_per_observable=0 should fail."
        )

    except ValueError:
        pass

    # ========================================================
    # Report
    # ========================================================

    if verbose:

        print("Cell 6 unit tests: PASS")

        print(
            "  ✓ Bell-state Pauli stabilizers are correct"
        )

        print(
            "  ✓ product-state Pauli expectations are correct"
        )

        print(
            "  ✓ exact Pauli tomography reconstructs pure states"
        )

        print(
            "  ✓ exact Pauli tomography reconstructs noisy mixed states"
        )

        print(
            "  ✓ finite-shot sampling is reproducible with a fixed seed"
        )

        print(
            "  ✓ finite-shot Pauli estimates converge to exact values"
        )

        print(
            "  ✓ PSD projection returns a physical density matrix"
        )

        print(
            "  ✓ finite-shot Bell fidelity is recovered"
        )

        print(
            "  ✓ finite-shot Bell concurrence is recovered"
        )

        print(
            "  ✓ tomographic metrics are internally consistent"
        )

        print(
            "  ✓ invalid shot counts are rejected"
        )


# Run immediately
run_cell6_unit_tests()

Cell 6 unit tests: PASS
  ✓ Bell-state Pauli stabilizers are correct
  ✓ product-state Pauli expectations are correct
  ✓ exact Pauli tomography reconstructs pure states
  ✓ exact Pauli tomography reconstructs noisy mixed states
  ✓ finite-shot sampling is reproducible with a fixed seed
  ✓ finite-shot Pauli estimates converge to exact values
  ✓ PSD projection returns a physical density matrix
  ✓ finite-shot Bell fidelity is recovered
  ✓ finite-shot Bell concurrence is recovered
  ✓ tomographic metrics are internally consistent
  ✓ invalid shot counts are rejected


In [7]:
# ============================================================
# V2 — Cell 7
# Finite-shot hybrid optimizer
#
# Requires Cells 1–6.
#
# New capability:
#
#   classical candidate (alpha, phi)
#       ->
#   noisy quantum circuit
#       ->
#   finite-shot Pauli tomography
#       ->
#   reconstructed rho_hat
#       ->
#   estimated concurrence + fidelity
#       ->
#   classical objective
#       ->
#   next candidate
#
# Classical optimizer:
#   transparent coarse-to-fine derivative-free search
#
# IMPORTANT:
#   The optimizer uses ONLY estimated finite-shot metrics.
#   Exact metrics are retained only as simulator diagnostics.
# ============================================================


# ------------------------------------------------------------
# 1. Evaluate one candidate using finite-shot measurements
# ------------------------------------------------------------

def evaluate_candidate_finite_shot(
    alpha,
    phi,
    target_concurrence,
    target_phi=0.0,
    shots_per_observable=2000,
    seed=0,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Evaluate one circuit candidate using only finite-shot
    tomographic estimates for the control metrics.

    Exact-state quantities are returned ONLY for diagnostic
    comparison and are not used by the optimizer.
    """

    (
        alpha_target,
        phi_target,
        psi_target
    ) = target_from_concurrence_phase(
        target_concurrence,
        target_phi
    )

    # --------------------------------------------------------
    # "Quantum backend"
    # --------------------------------------------------------

    rho_true = prepare_noisy_density_matrix_phase(
        alpha=alpha,
        phi=phi,
        noise_after_ry=noise_after_ry,
        noise_after_cnot=noise_after_cnot
    )

    # --------------------------------------------------------
    # Finite-shot measurement
    # --------------------------------------------------------

    tomography = run_pauli_tomography(
        rho=rho_true,
        shots_per_observable=shots_per_observable,
        seed=seed
    )

    rho_est = tomography[
        "rho_physical"
    ]

    # --------------------------------------------------------
    # Metrics available to the classical controller
    # --------------------------------------------------------

    C_est = concurrence_mixed(
        rho_est
    )

    F_est = fidelity_with_pure_target(
        rho_est,
        psi_target
    )

    # --------------------------------------------------------
    # Exact quantities:
    # simulator diagnostics only
    # --------------------------------------------------------

    C_true = concurrence_mixed(
        rho_true
    )

    F_true = fidelity_with_pure_target(
        rho_true,
        psi_target
    )

    return {
        "alpha": float(alpha),
        "phi": wrap_phase(phi),

        "target_alpha": alpha_target,
        "target_phi": phi_target,

        "rho_true": rho_true,
        "rho_est": rho_est,

        "estimated_concurrence": C_est,
        "estimated_fidelity": F_est,

        "true_concurrence": C_true,
        "true_fidelity": F_true,

        "shots_per_observable": int(
            shots_per_observable
        ),

        "shot_seed": int(seed),

        "tomography": tomography
    }


# ------------------------------------------------------------
# 2. Finite-shot objective
#
# The optimizer is allowed to use ONLY estimated quantities.
#
#   L =
#
#     w_C (C_hat - C_target)^2
#
#       +
#
#     w_F (1 - F_hat)
# ------------------------------------------------------------

def finite_shot_state_loss(
    result,
    target_concurrence,
    weight_concurrence=1.0,
    weight_fidelity=1.0
):
    """
    State-aware finite-shot objective.
    """

    weight_concurrence = float(
        weight_concurrence
    )

    weight_fidelity = float(
        weight_fidelity
    )

    if weight_concurrence < 0.0:
        raise ValueError(
            "weight_concurrence must be non-negative."
        )

    if weight_fidelity <= 0.0:
        raise ValueError(
            "weight_fidelity must be positive."
        )

    loss_C = (
        result["estimated_concurrence"]
        - target_concurrence
    ) ** 2

    loss_F = (
        1.0
        - result["estimated_fidelity"]
    )

    loss = (
        weight_concurrence * loss_C
        +
        weight_fidelity * loss_F
    )

    return {
        "loss": float(loss),
        "loss_C": float(loss_C),
        "loss_F": float(loss_F)
    }


# ------------------------------------------------------------
# 3. Coarse-to-fine finite-shot optimizer
#
# Stage 0:
#   global search over alpha and phi
#
# Later stages:
#   local refinement around the best measured candidate
#
# Every candidate uses an independent finite-shot sample.
# ------------------------------------------------------------

def run_finite_shot_hybrid_optimizer(
    target_concurrence,
    target_phi=0.0,
    shots_per_observable=2000,
    seed=3,

    n_alpha=17,
    n_phi=25,

    refinement_rounds=2,
    refinement_alpha_points=9,
    refinement_phi_points=13,

    weight_concurrence=1.0,
    weight_fidelity=1.0,

    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Finite-shot hybrid classical-quantum optimization.

    The search is deliberately simple and transparent.

    Returns:
        best result
        complete evaluation history
        shot budget
    """

    target_concurrence = float(
        target_concurrence
    )

    if not (
        0.0 <= target_concurrence <= 1.0
    ):
        raise ValueError(
            "target_concurrence must lie in [0, 1]."
        )

    n_alpha = int(
        n_alpha
    )

    n_phi = int(
        n_phi
    )

    refinement_rounds = int(
        refinement_rounds
    )

    if n_alpha < 2:
        raise ValueError(
            "n_alpha must be at least 2."
        )

    if n_phi < 4:
        raise ValueError(
            "n_phi must be at least 4."
        )

    if refinement_alpha_points < 3:
        raise ValueError(
            "refinement_alpha_points must be at least 3."
        )

    if refinement_phi_points < 3:
        raise ValueError(
            "refinement_phi_points must be at least 3."
        )

    rng = np.random.default_rng(
        seed
    )

    history = []

    best = None

    # --------------------------------------------------------
    # Internal candidate evaluator
    # --------------------------------------------------------

    def evaluate_point(
        alpha,
        phi,
        stage
    ):
        nonlocal best

        shot_seed = int(
            rng.integers(
                0,
                np.iinfo(
                    np.int32
                ).max
            )
        )

        result = evaluate_candidate_finite_shot(
            alpha=float(alpha),
            phi=float(phi),

            target_concurrence=target_concurrence,
            target_phi=target_phi,

            shots_per_observable=
                shots_per_observable,

            seed=shot_seed,

            noise_after_ry=noise_after_ry,
            noise_after_cnot=noise_after_cnot
        )

        loss_data = finite_shot_state_loss(
            result=result,
            target_concurrence=target_concurrence,
            weight_concurrence=
                weight_concurrence,
            weight_fidelity=
                weight_fidelity
        )

        result.update(
            loss_data
        )

        result["stage"] = int(
            stage
        )

        history.append(
            result
        )

        if (
            best is None
            or result["loss"]
            < best["loss"]
        ):
            best = result

    # --------------------------------------------------------
    # Stage 0:
    # global coarse search
    # --------------------------------------------------------

    alpha_grid = np.linspace(
        0.0,
        np.pi / 2.0,
        n_alpha
    )

    phi_grid = np.linspace(
        -np.pi,
        np.pi,
        n_phi,
        endpoint=False
    )

    for alpha in alpha_grid:

        for phi in phi_grid:

            evaluate_point(
                alpha,
                phi,
                stage=0
            )

    # Initial grid spacings
    delta_alpha = (
        alpha_grid[1]
        - alpha_grid[0]
    )

    delta_phi = (
        2.0 * np.pi
        / n_phi
    )

    # --------------------------------------------------------
    # Local refinement
    # --------------------------------------------------------

    for stage in range(
        1,
        refinement_rounds + 1
    ):

        alpha_center = best[
            "alpha"
        ]

        phi_center = best[
            "phi"
        ]

        alpha_grid = np.linspace(
            alpha_center
            - delta_alpha,

            alpha_center
            + delta_alpha,

            refinement_alpha_points
        )

        alpha_grid = np.clip(
            alpha_grid,
            0.0,
            np.pi / 2.0
        )

        phi_grid = np.linspace(
            phi_center
            - delta_phi,

            phi_center
            + delta_phi,

            refinement_phi_points
        )

        phi_grid = np.array([
            wrap_phase(phi)
            for phi in phi_grid
        ])

        for alpha in alpha_grid:

            for phi in phi_grid:

                evaluate_point(
                    alpha,
                    phi,
                    stage=stage
                )

        # New grid spacing for next refinement
        delta_alpha = (
            2.0
            * delta_alpha
            / (
                refinement_alpha_points
                - 1
            )
        )

        delta_phi = (
            2.0
            * delta_phi
            / (
                refinement_phi_points
                - 1
            )
        )

    # --------------------------------------------------------
    # Shot accounting
    #
    # 15 Pauli observables are measured per candidate.
    # --------------------------------------------------------

    number_of_evaluations = len(
        history
    )

    total_shots = (
        number_of_evaluations
        * len(PAULI_LABELS)
        * int(
            shots_per_observable
        )
    )

    return {
        "best": best,

        "history": history,

        "number_of_evaluations":
            number_of_evaluations,

        "shots_per_observable":
            int(
                shots_per_observable
            ),

        "total_shots":
            int(
                total_shots
            ),

        "target_concurrence":
            target_concurrence,

        "target_phi":
            wrap_phase(
                target_phi
            ),

        "weight_concurrence":
            float(
                weight_concurrence
            ),

        "weight_fidelity":
            float(
                weight_fidelity
            ),

        "seed":
            int(seed)
    }


# ------------------------------------------------------------
# 4. Cell 7 unit tests
# ------------------------------------------------------------

def run_cell7_unit_tests(verbose=True):

    C_target = 0.80
    phi_target = 0.40

    alpha_target = np.arcsin(
        C_target
    )

    # ========================================================
    # Test 1
    # One finite-shot candidate evaluation produces
    # valid reconstructed and exact states
    # ========================================================

    candidate = (
        evaluate_candidate_finite_shot(
            alpha=alpha_target,
            phi=phi_target,

            target_concurrence=C_target,
            target_phi=phi_target,

            shots_per_observable=2000,
            seed=123
        )
    )

    assert_density_matrix(
        candidate["rho_true"]
    )

    assert_density_matrix(
        candidate["rho_est"],
        atol=1e-10
    )

    # ========================================================
    # Test 2
    # The finite-shot objective uses ONLY estimated metrics
    # ========================================================

    loss_data = finite_shot_state_loss(
        result=candidate,
        target_concurrence=C_target,
        weight_concurrence=1.0,
        weight_fidelity=1.0
    )

    expected_loss = (
        (
            candidate[
                "estimated_concurrence"
            ]
            - C_target
        ) ** 2
        +
        (
            1.0
            - candidate[
                "estimated_fidelity"
            ]
        )
    )

    assert np.isclose(
        loss_data["loss"],
        expected_loss,
        atol=1e-14
    )

    # ========================================================
    # Test 3
    # Full finite-shot optimization
    # ========================================================

    result = run_finite_shot_hybrid_optimizer(
        target_concurrence=C_target,
        target_phi=phi_target,

        shots_per_observable=2000,

        seed=3,

        n_alpha=17,
        n_phi=25,

        refinement_rounds=2,

        weight_concurrence=1.0,
        weight_fidelity=1.0
    )

    best = result[
        "best"
    ]

    # ========================================================
    # Test 4
    # Search converges close to the target controls
    # ========================================================

    assert abs(
        best["alpha"]
        - alpha_target
    ) < 0.10

    assert abs(
        phase_distance(
            best["phi"],
            phi_target
        )
    ) < 0.20

    # ========================================================
    # Test 5
    # True state corresponding to selected controls
    # is very close to the target
    #
    # IMPORTANT:
    # This is a simulator diagnostic only.
    # The optimizer did not use this value.
    # ========================================================

    assert (
        best["true_fidelity"]
        > 0.995
    ), (
        "Finite-shot controller failed to find "
        "a high-fidelity physical state."
    )

    assert abs(
        best["true_concurrence"]
        - C_target
    ) < 0.05

    # ========================================================
    # Test 6
    # Estimated result is also high quality
    # ========================================================

    assert (
        best["estimated_fidelity"]
        > 0.98
    )

    assert abs(
        best["estimated_concurrence"]
        - C_target
    ) < 0.05

    # ========================================================
    # Test 7
    # Fixed random seed gives reproducible optimization
    # ========================================================

    result_repeat = (
        run_finite_shot_hybrid_optimizer(
            target_concurrence=C_target,
            target_phi=phi_target,

            shots_per_observable=2000,

            seed=3,

            n_alpha=17,
            n_phi=25,

            refinement_rounds=2,

            weight_concurrence=1.0,
            weight_fidelity=1.0
        )
    )

    best_repeat = result_repeat[
        "best"
    ]

    assert np.isclose(
        best["alpha"],
        best_repeat["alpha"],
        atol=1e-14
    )

    assert np.isclose(
        best["phi"],
        best_repeat["phi"],
        atol=1e-14
    )

    assert np.isclose(
        best["loss"],
        best_repeat["loss"],
        atol=1e-14
    )

    # ========================================================
    # Test 8
    # Shot accounting is correct
    # ========================================================

    assert (
        result["total_shots"]
        ==
        result["number_of_evaluations"]
        * len(PAULI_LABELS)
        * result["shots_per_observable"]
    )

    # ========================================================
    # Test 9
    # Every accepted candidate has a physical
    # reconstructed density matrix
    # ========================================================

    assert_density_matrix(
        best["rho_est"],
        atol=1e-10
    )

    # ========================================================
    # Report
    # ========================================================

    if verbose:

        print("Cell 7 unit tests: PASS")

        print(
            "  ✓ finite-shot candidate evaluation returns physical states"
        )

        print(
            "  ✓ controller objective uses estimated metrics only"
        )

        print(
            "  ✓ finite-shot hybrid search converges near target controls"
        )

        print(
            "  ✓ selected physical state has high true fidelity"
        )

        print(
            "  ✓ estimated concurrence and fidelity are consistent with target"
        )

        print(
            "  ✓ fixed seed makes the complete optimization reproducible"
        )

        print(
            "  ✓ finite-shot measurement budget is tracked explicitly"
        )

        print(
            "  ✓ reconstructed optimizer output remains physical"
        )

        print()
        print("Best finite-shot result:")
        print(
            f"  alpha target = {alpha_target:.6f}"
        )
        print(
            f"  alpha found  = {best['alpha']:.6f}"
        )
        print(
            f"  phi target   = {phi_target:.6f}"
        )
        print(
            f"  phi found    = {best['phi']:.6f}"
        )
        print(
            f"  C estimated  = "
            f"{best['estimated_concurrence']:.6f}"
        )
        print(
            f"  C true       = "
            f"{best['true_concurrence']:.6f}"
        )
        print(
            f"  F estimated  = "
            f"{best['estimated_fidelity']:.6f}"
        )
        print(
            f"  F true       = "
            f"{best['true_fidelity']:.6f}"
        )
        print(
            f"  evaluations  = "
            f"{result['number_of_evaluations']}"
        )
        print(
            f"  total shots  = "
            f"{result['total_shots']:,}"
        )


# Run immediately
run_cell7_unit_tests()

Cell 7 unit tests: PASS
  ✓ finite-shot candidate evaluation returns physical states
  ✓ controller objective uses estimated metrics only
  ✓ finite-shot hybrid search converges near target controls
  ✓ selected physical state has high true fidelity
  ✓ estimated concurrence and fidelity are consistent with target
  ✓ fixed seed makes the complete optimization reproducible
  ✓ finite-shot measurement budget is tracked explicitly
  ✓ reconstructed optimizer output remains physical

Best finite-shot result:
  alpha target = 0.927295
  alpha found  = 0.963340
  phi target   = 0.400000
  phi found    = 0.439823
  C estimated  = 0.795657
  C true       = 0.821103
  F estimated  = 0.994443
  F true       = 0.999415
  evaluations  = 659
  total shots  = 19,770,000


In [8]:
# ============================================================
# V2 — Cell 8
# Recoverable control errors vs irreducible quantum noise
#
# Requires Cells 1–7.
#
# New capability:
#
#   commanded controls
#       ->
#   systematic hardware calibration offsets
#       ->
#   noisy quantum circuit
#       ->
#   finite-shot measurements
#       ->
#   hybrid retuning
#
# We compare:
#
#   1. ideal target
#   2. nominal hardware operation
#   3. finite-shot hybrid retuning
#   4. exact noisy-hardware optimum
#
# This separates:
#
#   recoverable coherent error
#
# from
#
#   irreducible incoherent degradation.
# ============================================================


# ------------------------------------------------------------
# 1. Hardware-control bias
# ------------------------------------------------------------

def normalize_control_bias(
    control_bias=None
):
    """
    Normalize systematic control offsets.

    alpha_actual = alpha_command + alpha_offset
    phi_actual   = phi_command   + phi_offset
    """

    if control_bias is None:
        control_bias = {}

    return {
        "alpha_offset": float(
            control_bias.get(
                "alpha_offset",
                0.0
            )
        ),

        "phi_offset": float(
            control_bias.get(
                "phi_offset",
                0.0
            )
        )
    }


# ------------------------------------------------------------
# 2. Hardware implementation
# ------------------------------------------------------------

def prepare_hardware_density_matrix_phase(
    alpha_command,
    phi_command,
    control_bias=None,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Apply commanded controls to an imperfect device.

    The hardware implements:

        alpha_actual =
            alpha_command + delta_alpha

        phi_actual =
            phi_command + delta_phi

    before the previously validated noise model is applied.
    """

    bias = normalize_control_bias(
        control_bias
    )

    alpha_actual = (
        float(alpha_command)
        +
        bias["alpha_offset"]
    )

    phi_actual = (
        float(phi_command)
        +
        bias["phi_offset"]
    )

    rho = prepare_noisy_density_matrix_phase(
        alpha=alpha_actual,
        phi=phi_actual,
        noise_after_ry=noise_after_ry,
        noise_after_cnot=noise_after_cnot
    )

    return rho


# ------------------------------------------------------------
# 3. Exact hardware evaluation
#
# These values are available only to the simulator,
# not to the finite-shot controller.
# ------------------------------------------------------------

def evaluate_hardware_exact(
    alpha_command,
    phi_command,
    target_concurrence,
    target_phi=0.0,
    control_bias=None,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Exact simulator-level evaluation of imperfect hardware.
    """

    (
        alpha_target,
        phi_target,
        psi_target
    ) = target_from_concurrence_phase(
        target_concurrence,
        target_phi
    )

    rho = prepare_hardware_density_matrix_phase(
        alpha_command=alpha_command,
        phi_command=phi_command,
        control_bias=control_bias,
        noise_after_ry=noise_after_ry,
        noise_after_cnot=noise_after_cnot
    )

    C = concurrence_mixed(
        rho
    )

    F = fidelity_with_pure_target(
        rho,
        psi_target
    )

    return {
        "rho": rho,

        "alpha_command": float(
            alpha_command
        ),

        "phi_command": wrap_phase(
            phi_command
        ),

        "target_alpha": alpha_target,
        "target_phi": phi_target,

        "concurrence": C,

        "concurrence_error": abs(
            C - target_concurrence
        ),

        "fidelity": F,

        "infidelity": 1.0 - F,

        "purity": purity(
            rho
        )
    }


# ------------------------------------------------------------
# 4. Finite-shot hardware evaluation
#
# This is the quantity visible to the controller.
# ------------------------------------------------------------

def evaluate_hardware_candidate_finite_shot(
    alpha_command,
    phi_command,
    target_concurrence,
    target_phi=0.0,
    shots_per_observable=2000,
    seed=0,
    control_bias=None,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Evaluate imperfect hardware through finite-shot tomography.
    """

    (
        alpha_target,
        phi_target,
        psi_target
    ) = target_from_concurrence_phase(
        target_concurrence,
        target_phi
    )

    rho_true = (
        prepare_hardware_density_matrix_phase(
            alpha_command=alpha_command,
            phi_command=phi_command,
            control_bias=control_bias,
            noise_after_ry=noise_after_ry,
            noise_after_cnot=noise_after_cnot
        )
    )

    tomography = run_pauli_tomography(
        rho=rho_true,
        shots_per_observable=
            shots_per_observable,
        seed=seed
    )

    rho_est = tomography[
        "rho_physical"
    ]

    return {
        "alpha": float(
            alpha_command
        ),

        "phi": wrap_phase(
            phi_command
        ),

        "rho_true": rho_true,
        "rho_est": rho_est,

        # Controller-visible quantities
        "estimated_concurrence":
            concurrence_mixed(
                rho_est
            ),

        "estimated_fidelity":
            fidelity_with_pure_target(
                rho_est,
                psi_target
            ),

        # Simulator diagnostics only
        "true_concurrence":
            concurrence_mixed(
                rho_true
            ),

        "true_fidelity":
            fidelity_with_pure_target(
                rho_true,
                psi_target
            ),

        "shots_per_observable":
            int(
                shots_per_observable
            ),

        "seed":
            int(seed)
    }


# ------------------------------------------------------------
# 5. Exact noisy-hardware optimum
#
# This is an oracle calculation.
#
# It answers:
#
#     If we knew the hardware exactly,
#     what is the best state this hardware could make?
#
# It is NOT available to the hybrid controller.
# ------------------------------------------------------------

def run_exact_hardware_optimizer(
    target_concurrence,
    target_phi=0.0,
    control_bias=None,
    noise_after_ry=None,
    noise_after_cnot=None,
    weight_concurrence=1.0,
    weight_fidelity=1.0,
    maxiter=500
):
    """
    Compute the exact noisy-hardware ceiling.
    """

    (
        alpha_target,
        phi_target,
        psi_target
    ) = target_from_concurrence_phase(
        target_concurrence,
        target_phi
    )

    def objective(x):

        result = evaluate_hardware_exact(
            alpha_command=x[0],
            phi_command=x[1],

            target_concurrence=
                target_concurrence,

            target_phi=
                target_phi,

            control_bias=
                control_bias,

            noise_after_ry=
                noise_after_ry,

            noise_after_cnot=
                noise_after_cnot
        )

        loss_C = (
            result[
                "concurrence_error"
            ] ** 2
        )

        loss_F = (
            1.0
            - result[
                "fidelity"
            ]
        )

        return (
            weight_concurrence
            * loss_C
            +
            weight_fidelity
            * loss_F
        )

    opt = minimize(
        objective,

        x0=np.array([
            alpha_target,
            phi_target
        ]),

        method="Powell",

        bounds=[
            (
                0.0,
                np.pi / 2.0
            ),
            (
                -np.pi,
                np.pi
            )
        ],

        options={
            "xtol": 1e-12,
            "ftol": 1e-14,
            "maxiter": int(
                maxiter
            )
        }
    )

    result = evaluate_hardware_exact(
        alpha_command=
            float(
                opt.x[0]
            ),

        phi_command=
            wrap_phase(
                opt.x[1]
            ),

        target_concurrence=
            target_concurrence,

        target_phi=
            target_phi,

        control_bias=
            control_bias,

        noise_after_ry=
            noise_after_ry,

        noise_after_cnot=
            noise_after_cnot
    )

    result.update({
        "optimizer_success":
            bool(
                opt.success
            ),

        "loss":
            float(
                opt.fun
            ),

        "alpha_opt":
            float(
                opt.x[0]
            ),

        "phi_opt":
            wrap_phase(
                opt.x[1]
            )
    })

    return result


# ------------------------------------------------------------
# 6. Finite-shot hardware optimizer
#
# Same transparent coarse-to-fine strategy as Cell 7.
#
# The controller only sees:
#
#     estimated concurrence
#     estimated fidelity
# ------------------------------------------------------------

def run_finite_shot_hardware_optimizer(
    target_concurrence,
    target_phi=0.0,

    shots_per_observable=2000,
    seed=3,

    n_alpha=17,
    n_phi=25,

    refinement_rounds=2,
    refinement_alpha_points=9,
    refinement_phi_points=13,

    weight_concurrence=1.0,
    weight_fidelity=1.0,

    control_bias=None,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Finite-shot hybrid retuning of imperfect hardware.
    """

    rng = np.random.default_rng(
        seed
    )

    history = []

    best = None

    def evaluate_point(
        alpha,
        phi,
        stage
    ):
        nonlocal best

        shot_seed = int(
            rng.integers(
                0,
                np.iinfo(
                    np.int32
                ).max
            )
        )

        result = (
            evaluate_hardware_candidate_finite_shot(
                alpha_command=
                    float(
                        alpha
                    ),

                phi_command=
                    float(
                        phi
                    ),

                target_concurrence=
                    target_concurrence,

                target_phi=
                    target_phi,

                shots_per_observable=
                    shots_per_observable,

                seed=
                    shot_seed,

                control_bias=
                    control_bias,

                noise_after_ry=
                    noise_after_ry,

                noise_after_cnot=
                    noise_after_cnot
            )
        )

        loss_C = (
            result[
                "estimated_concurrence"
            ]
            -
            target_concurrence
        ) ** 2

        loss_F = (
            1.0
            -
            result[
                "estimated_fidelity"
            ]
        )

        loss = (
            weight_concurrence
            * loss_C
            +
            weight_fidelity
            * loss_F
        )

        result.update({
            "loss":
                float(
                    loss
                ),

            "loss_C":
                float(
                    loss_C
                ),

            "loss_F":
                float(
                    loss_F
                ),

            "stage":
                int(
                    stage
                )
        })

        history.append(
            result
        )

        if (
            best is None
            or result[
                "loss"
            ]
            <
            best[
                "loss"
            ]
        ):
            best = result

    # --------------------------------------------------------
    # Global search
    # --------------------------------------------------------

    alpha_grid = np.linspace(
        0.0,
        np.pi / 2.0,
        n_alpha
    )

    phi_grid = np.linspace(
        -np.pi,
        np.pi,
        n_phi,
        endpoint=False
    )

    for alpha in alpha_grid:

        for phi in phi_grid:

            evaluate_point(
                alpha,
                phi,
                stage=0
            )

    delta_alpha = (
        alpha_grid[1]
        - alpha_grid[0]
    )

    delta_phi = (
        2.0 * np.pi
        / n_phi
    )

    # --------------------------------------------------------
    # Refinement
    # --------------------------------------------------------

    for stage in range(
        1,
        refinement_rounds + 1
    ):

        alpha_center = best[
            "alpha"
        ]

        phi_center = best[
            "phi"
        ]

        alpha_grid = np.linspace(
            alpha_center
            - delta_alpha,

            alpha_center
            + delta_alpha,

            refinement_alpha_points
        )

        alpha_grid = np.clip(
            alpha_grid,
            0.0,
            np.pi / 2.0
        )

        phi_grid = np.linspace(
            phi_center
            - delta_phi,

            phi_center
            + delta_phi,

            refinement_phi_points
        )

        phi_grid = np.array([
            wrap_phase(
                phi
            )
            for phi in phi_grid
        ])

        for alpha in alpha_grid:

            for phi in phi_grid:

                evaluate_point(
                    alpha,
                    phi,
                    stage=stage
                )

        delta_alpha = (
            2.0
            * delta_alpha
            /
            (
                refinement_alpha_points
                - 1
            )
        )

        delta_phi = (
            2.0
            * delta_phi
            /
            (
                refinement_phi_points
                - 1
            )
        )

    number_of_evaluations = len(
        history
    )

    total_shots = (
        number_of_evaluations
        *
        len(
            PAULI_LABELS
        )
        *
        int(
            shots_per_observable
        )
    )

    return {
        "best": best,

        "history": history,

        "number_of_evaluations":
            number_of_evaluations,

        "total_shots":
            int(
                total_shots
            ),

        "shots_per_observable":
            int(
                shots_per_observable
            ),

        "seed":
            int(
                seed
            )
    }


# ------------------------------------------------------------
# 7. Recoverability analysis
# ------------------------------------------------------------

def build_recovery_summary(
    target_concurrence,
    target_phi,

    nominal,
    controlled,
    hardware_ceiling
):
    """
    Separate fidelity degradation into:

        recoverable opportunity
        recovered amount
        controller gap
        irreducible physical gap
    """

    F_ideal = 1.0

    F_nominal = float(
        nominal[
            "fidelity"
        ]
    )

    F_controlled = float(
        controlled[
            "true_fidelity"
        ]
    )

    F_ceiling = float(
        hardware_ceiling[
            "fidelity"
        ]
    )

    recoverable_opportunity = max(
        0.0,
        F_ceiling
        - F_nominal
    )

    recovered = (
        F_controlled
        - F_nominal
    )

    controller_gap = (
        F_ceiling
        - F_controlled
    )

    irreducible_gap = (
        F_ideal
        - F_ceiling
    )

    if (
        recoverable_opportunity
        > 1e-15
    ):

        recovered_fraction = (
            recovered
            /
            recoverable_opportunity
        )

    else:

        recovered_fraction = 1.0

    return {
        "F_ideal":
            F_ideal,

        "F_nominal":
            F_nominal,

        "F_controlled":
            F_controlled,

        "F_hardware_ceiling":
            F_ceiling,

        "recoverable_opportunity":
            recoverable_opportunity,

        "recovered":
            recovered,

        "recovered_fraction":
            recovered_fraction,

        "controller_gap":
            controller_gap,

        "irreducible_gap":
            irreducible_gap
    }


# ------------------------------------------------------------
# 8. Cell 8 unit tests
# ------------------------------------------------------------

def run_cell8_unit_tests(
    verbose=True
):

    # ========================================================
    # Benchmark:
    #
    # target:
    #     Bell state
    #
    # coherent hardware error:
    #     delta alpha = +0.25
    #     delta phi   = +0.70
    #
    # irreversible hardware noise:
    #     one-sided post-CNOT dephasing p = 0.20
    #
    # Analytic ceilings:
    #
    #     C_max = 1-p   = 0.80
    #
    #     F_max = 1-p/2 = 0.90
    # ========================================================

    C_target = 1.0
    phi_target = 0.0

    alpha_target = (
        np.pi
        / 2.0
    )

    control_bias = {
        "alpha_offset":
            0.25,

        "phi_offset":
            0.70
    }

    noise_after_cnot = {
        "dephasing": (
            0.20,
            0.0
        )
    }

    # ========================================================
    # Test 1
    # No-bias / no-noise hardware wrapper reproduces
    # the frozen ideal circuit.
    # ========================================================

    rho_wrapper = (
        prepare_hardware_density_matrix_phase(
            alpha_command=0.83,
            phi_command=0.41
        )
    )

    rho_ideal = (
        prepare_ideal_density_matrix_phase(
            alpha=0.83,
            phi=0.41
        )
    )

    assert np.allclose(
        rho_wrapper,
        rho_ideal,
        atol=1e-12
    )

    # ========================================================
    # Test 2
    # Coherent offsets alone are exactly recoverable.
    # ========================================================

    alpha_corrected = (
        alpha_target
        -
        control_bias[
            "alpha_offset"
        ]
    )

    phi_corrected = (
        phi_target
        -
        control_bias[
            "phi_offset"
        ]
    )

    corrected_no_noise = (
        evaluate_hardware_exact(
            alpha_command=
                alpha_corrected,

            phi_command=
                phi_corrected,

            target_concurrence=
                C_target,

            target_phi=
                phi_target,

            control_bias=
                control_bias
        )
    )

    assert np.isclose(
        corrected_no_noise[
            "fidelity"
        ],
        1.0,
        atol=1e-12
    )

    assert np.isclose(
        corrected_no_noise[
            "concurrence"
        ],
        1.0,
        atol=1e-8
    )

    # ========================================================
    # Test 3
    # With dephasing present, corrected controls reach
    # the known physical ceiling:
    #
    #     C = 0.8
    #     F = 0.9
    # ========================================================

    corrected_with_noise = (
        evaluate_hardware_exact(
            alpha_command=
                alpha_corrected,

            phi_command=
                phi_corrected,

            target_concurrence=
                C_target,

            target_phi=
                phi_target,

            control_bias=
                control_bias,

            noise_after_cnot=
                noise_after_cnot
        )
    )

    assert np.isclose(
        corrected_with_noise[
            "concurrence"
        ],
        0.80,
        atol=1e-8
    )

    assert np.isclose(
        corrected_with_noise[
            "fidelity"
        ],
        0.90,
        atol=1e-12
    )

    # ========================================================
    # Test 4
    # Exact optimizer finds the noisy-hardware ceiling.
    # ========================================================

    exact_optimum = (
        run_exact_hardware_optimizer(
            target_concurrence=
                C_target,

            target_phi=
                phi_target,

            control_bias=
                control_bias,

            noise_after_cnot=
                noise_after_cnot
        )
    )

    assert (
        exact_optimum[
            "optimizer_success"
        ]
        is True
    )

    assert np.isclose(
        exact_optimum[
            "fidelity"
        ],
        0.90,
        atol=1e-8
    )

    assert np.isclose(
        exact_optimum[
            "concurrence"
        ],
        0.80,
        atol=1e-7
    )

    assert abs(
        exact_optimum[
            "alpha_opt"
        ]
        -
        alpha_corrected
    ) < 1e-5

    assert abs(
        phase_distance(
            exact_optimum[
                "phi_opt"
            ],
            phi_corrected
        )
    ) < 1e-5

    # ========================================================
    # Test 5
    # Nominal hardware operation is degraded by both
    # coherent offsets and dephasing.
    # ========================================================

    nominal = (
        evaluate_hardware_exact(
            alpha_command=
                alpha_target,

            phi_command=
                phi_target,

            target_concurrence=
                C_target,

            target_phi=
                phi_target,

            control_bias=
                control_bias,

            noise_after_cnot=
                noise_after_cnot
        )
    )

    assert (
        nominal[
            "fidelity"
        ]
        <
        exact_optimum[
            "fidelity"
        ]
    )

    # ========================================================
    # Test 6
    # Finite-shot hybrid controller recovers almost all
    # coherent degradation.
    # ========================================================

    hybrid = (
        run_finite_shot_hardware_optimizer(
            target_concurrence=
                C_target,

            target_phi=
                phi_target,

            shots_per_observable=
                2000,

            seed=
                3,

            n_alpha=
                17,

            n_phi=
                25,

            refinement_rounds=
                2,

            control_bias=
                control_bias,

            noise_after_cnot=
                noise_after_cnot
        )
    )

    best = hybrid[
        "best"
    ]

    assert (
        best[
            "true_fidelity"
        ]
        >
        0.895
    )

    assert abs(
        best[
            "true_concurrence"
        ]
        -
        0.80
    ) < 0.01

    # ========================================================
    # Test 7
    # Finite-shot solution lies close to corrected controls.
    # ========================================================

    assert abs(
        best[
            "alpha"
        ]
        -
        alpha_corrected
    ) < 0.10

    assert abs(
        phase_distance(
            best[
                "phi"
            ],
            phi_corrected
        )
    ) < 0.15

    # ========================================================
    # Test 8
    # Controller does NOT exceed physical hardware ceiling.
    # ========================================================

    assert (
        best[
            "true_fidelity"
        ]
        <=
        exact_optimum[
            "fidelity"
        ]
        + 1e-10
    )

    # ========================================================
    # Test 9
    # Recoverability decomposition.
    # ========================================================

    recovery = (
        build_recovery_summary(
            target_concurrence=
                C_target,

            target_phi=
                phi_target,

            nominal=
                nominal,

            controlled=
                best,

            hardware_ceiling=
                exact_optimum
        )
    )

    assert (
        recovery[
            "recoverable_opportunity"
        ]
        >
        0.0
    )

    assert (
        recovery[
            "recovered"
        ]
        >
        0.0
    )

    assert (
        recovery[
            "recovered_fraction"
        ]
        >
        0.95
    )

    assert np.isclose(
        recovery[
            "irreducible_gap"
        ],
        0.10,
        atol=1e-8
    )

    # ========================================================
    # Test 10
    # Fidelity decomposition closes.
    #
    # ideal - nominal =
    #
    #   recoverable opportunity
    #   +
    #   irreducible gap
    # ========================================================

    assert np.isclose(
        (
            recovery[
                "F_ideal"
            ]
            -
            recovery[
                "F_nominal"
            ]
        ),

        (
            recovery[
                "recoverable_opportunity"
            ]
            +
            recovery[
                "irreducible_gap"
            ]
        ),

        atol=1e-12
    )

    # ========================================================
    # Test 11
    # Finite-shot accounting remains correct.
    # ========================================================

    assert (
        hybrid[
            "total_shots"
        ]
        ==
        hybrid[
            "number_of_evaluations"
        ]
        *
        len(
            PAULI_LABELS
        )
        *
        hybrid[
            "shots_per_observable"
        ]
    )

    # ========================================================
    # Report
    # ========================================================

    if verbose:

        print(
            "Cell 8 unit tests: PASS"
        )

        print(
            "  ✓ hardware wrapper reproduces ideal limit"
        )

        print(
            "  ✓ coherent control offsets are fully recoverable"
        )

        print(
            "  ✓ dephasing imposes analytic C=0.80 and F=0.90 ceilings"
        )

        print(
            "  ✓ exact optimizer finds the physical hardware ceiling"
        )

        print(
            "  ✓ nominal operation is degraded by calibration error + noise"
        )

        print(
            "  ✓ finite-shot hybrid controller recovers coherent error"
        )

        print(
            "  ✓ recovered controls agree with expected calibration correction"
        )

        print(
            "  ✓ hybrid controller does not exceed the noisy-hardware ceiling"
        )

        print(
            "  ✓ recoverable and irreducible degradation are separated"
        )

        print(
            "  ✓ fidelity degradation decomposition closes"
        )

        print(
            "  ✓ finite-shot budget remains explicit"
        )

        print()

        print(
            "Deep-tech benchmark:"
        )

        print(
            f"  nominal fidelity       = "
            f"{nominal['fidelity']:.6f}"
        )

        print(
            f"  controlled fidelity    = "
            f"{best['true_fidelity']:.6f}"
        )

        print(
            f"  hardware ceiling       = "
            f"{exact_optimum['fidelity']:.6f}"
        )

        print(
            f"  ideal fidelity          = "
            f"{1.0:.6f}"
        )

        print()

        print(
            f"  nominal concurrence    = "
            f"{nominal['concurrence']:.6f}"
        )

        print(
            f"  controlled concurrence = "
            f"{best['true_concurrence']:.6f}"
        )

        print(
            f"  hardware C ceiling     = "
            f"{exact_optimum['concurrence']:.6f}"
        )

        print()

        print(
            f"  alpha corrected target = "
            f"{alpha_corrected:.6f}"
        )

        print(
            f"  alpha controller found = "
            f"{best['alpha']:.6f}"
        )

        print(
            f"  phi corrected target   = "
            f"{phi_corrected:.6f}"
        )

        print(
            f"  phi controller found   = "
            f"{best['phi']:.6f}"
        )

        print()

        print(
            f"  recoverable fidelity   = "
            f"{recovery['recoverable_opportunity']:.6f}"
        )

        print(
            f"  fidelity recovered     = "
            f"{recovery['recovered']:.6f}"
        )

        print(
            f"  recovery fraction      = "
            f"{100.0 * recovery['recovered_fraction']:.1f}%"
        )

        print(
            f"  controller gap         = "
            f"{recovery['controller_gap']:.6f}"
        )

        print(
            f"  irreducible gap        = "
            f"{recovery['irreducible_gap']:.6f}"
        )

        print()

        print(
            f"  evaluations            = "
            f"{hybrid['number_of_evaluations']}"
        )

        print(
            f"  total shots            = "
            f"{hybrid['total_shots']:,}"
        )


# Run immediately
run_cell8_unit_tests()

Cell 8 unit tests: PASS
  ✓ hardware wrapper reproduces ideal limit
  ✓ coherent control offsets are fully recoverable
  ✓ dephasing imposes analytic C=0.80 and F=0.90 ceilings
  ✓ exact optimizer finds the physical hardware ceiling
  ✓ nominal operation is degraded by calibration error + noise
  ✓ finite-shot hybrid controller recovers coherent error
  ✓ recovered controls agree with expected calibration correction
  ✓ hybrid controller does not exceed the noisy-hardware ceiling
  ✓ recoverable and irreducible degradation are separated
  ✓ fidelity degradation decomposition closes
  ✓ finite-shot budget remains explicit

Deep-tech benchmark:
  nominal fidelity       = 0.796426
  controlled fidelity    = 0.898022
  hardware ceiling       = 0.900000
  ideal fidelity          = 1.000000

  nominal concurrence    = 0.775130
  controlled concurrence = 0.798093
  hardware C ceiling     = 0.800000

  alpha corrected target = 1.320796
  alpha controller found = 1.251728
  phi corrected target

In [9]:
# ============================================================
# V2 — Cell 9
# Model-aware measurement compression for X states
#
# Requires Cells 1–8.
#
# Key result:
#
#   Full two-qubit tomography:
#       15 nontrivial Pauli observables
#
#   Current V2 hardware/noise model:
#       state remains in X-state family
#
#   Therefore only 7 observables are required:
#
#       ZI, IZ, ZZ,
#       XX, YY, XY, YX
#
# This is NOT universal tomography.
# It is model-aware tomography for the validated state family.
# ============================================================


# ------------------------------------------------------------
# 1. Reduced measurement set
# ------------------------------------------------------------

X_STATE_LABELS = [
    "ZI",
    "IZ",
    "ZZ",
    "XX",
    "YY",
    "XY",
    "YX"
]


# ------------------------------------------------------------
# 2. Measure deviation from X-state structure
#
# Allowed nonzero matrix entries:
#
#   diagonal
#   rho_03, rho_30
#   rho_12, rho_21
# ------------------------------------------------------------

def x_state_leakage(rho):
    """
    Maximum absolute matrix element outside the
    two-qubit X-state structure.
    """

    rho = np.asarray(
        rho,
        dtype=complex
    )

    if rho.shape != (4, 4):
        raise ValueError(
            "rho must be a 4x4 density matrix."
        )

    allowed = np.zeros(
        (4, 4),
        dtype=bool
    )

    for i in range(4):
        allowed[i, i] = True

    allowed[0, 3] = True
    allowed[3, 0] = True

    allowed[1, 2] = True
    allowed[2, 1] = True

    forbidden = rho[
        ~allowed
    ]

    if forbidden.size == 0:
        return 0.0

    return float(
        np.max(
            np.abs(
                forbidden
            )
        )
    )


# ------------------------------------------------------------
# 3. Reconstruct an X state from seven Pauli expectations
#
# Diagonal populations:
#
#   a = rho_00
#   b = rho_11
#   c = rho_22
#   d = rho_33
#
# determined by:
#
#   ZI, IZ, ZZ
#
# Coherences:
#
#   z = rho_03
#   w = rho_12
#
# determined by:
#
#   XX, YY, XY, YX
# ------------------------------------------------------------

def rho_from_x_expectations(
    expectations
):
    """
    Reconstruct the complete two-qubit X-state density matrix
    from seven Pauli expectation values.
    """

    missing = [
        label
        for label in X_STATE_LABELS
        if label not in expectations
    ]

    if missing:
        raise ValueError(
            "Missing X-state observables: "
            + ", ".join(missing)
        )

    ZI = float(
        expectations["ZI"]
    )

    IZ = float(
        expectations["IZ"]
    )

    ZZ = float(
        expectations["ZZ"]
    )

    XX = float(
        expectations["XX"]
    )

    YY = float(
        expectations["YY"]
    )

    XY = float(
        expectations["XY"]
    )

    YX = float(
        expectations["YX"]
    )

    # --------------------------------------------------------
    # Populations
    # --------------------------------------------------------

    a = (
        1.0
        + ZI
        + IZ
        + ZZ
    ) / 4.0

    b = (
        1.0
        + ZI
        - IZ
        - ZZ
    ) / 4.0

    c = (
        1.0
        - ZI
        + IZ
        - ZZ
    ) / 4.0

    d = (
        1.0
        - ZI
        - IZ
        + ZZ
    ) / 4.0

    # --------------------------------------------------------
    # Coherence z = rho_03
    #
    # XX = 2 Re(z+w)
    # YY = 2 Re(w-z)
    #
    # XY = 2[Im(w)-Im(z)]
    # YX = -2[Im(z)+Im(w)]
    # --------------------------------------------------------

    Re_z = (
        XX - YY
    ) / 4.0

    Re_w = (
        XX + YY
    ) / 4.0

    Im_z = -(
        XY + YX
    ) / 4.0

    Im_w = (
        XY - YX
    ) / 4.0

    z = (
        Re_z
        + 1j * Im_z
    )

    w = (
        Re_w
        + 1j * Im_w
    )

    rho = np.zeros(
        (4, 4),
        dtype=complex
    )

    rho[0, 0] = a
    rho[1, 1] = b
    rho[2, 2] = c
    rho[3, 3] = d

    rho[0, 3] = z
    rho[3, 0] = np.conj(
        z
    )

    rho[1, 2] = w
    rho[2, 1] = np.conj(
        w
    )

    rho = (
        rho
        + rho.conj().T
    ) / 2.0

    return rho


# ------------------------------------------------------------
# 4. Exact seven-observable table
# ------------------------------------------------------------

def exact_x_state_table(
    rho
):
    """
    Exact expectation values for the seven observables
    required to reconstruct an X state.
    """

    return {
        label: pauli_expectation(
            rho,
            label
        )
        for label in X_STATE_LABELS
    }


# ------------------------------------------------------------
# 5. Finite-shot seven-observable measurement
# ------------------------------------------------------------

def sample_x_state_table(
    rho,
    shots_per_observable=2000,
    seed=0
):
    """
    Sample only the seven observables required by the
    X-state model.
    """

    shots_per_observable = int(
        shots_per_observable
    )

    if shots_per_observable <= 0:
        raise ValueError(
            "shots_per_observable must be positive."
        )

    rng = np.random.default_rng(
        seed
    )

    records = {}

    for label in X_STATE_LABELS:

        records[label] = (
            sample_pauli_expectation(
                rho=rho,
                label=label,
                shots=shots_per_observable,
                rng=rng
            )
        )

    return records


# ------------------------------------------------------------
# 6. Reduced X-state tomography
# ------------------------------------------------------------

def run_x_state_tomography(
    rho,
    shots_per_observable=2000,
    seed=0
):
    """
    Perform finite-shot model-aware X-state tomography.
    """

    records = sample_x_state_table(
        rho=rho,
        shots_per_observable=
            shots_per_observable,
        seed=seed
    )

    expectations = {
        label:
            records[label]["estimate"]

        for label
        in X_STATE_LABELS
    }

    rho_raw = (
        rho_from_x_expectations(
            expectations
        )
    )

    # As in Cell 6, finite-shot linear inversion can produce
    # small negative eigenvalues.
    rho_physical = (
        project_to_physical_density_matrix(
            rho_raw
        )
    )

    return {
        "records":
            records,

        "expectations":
            expectations,

        "rho_raw":
            rho_raw,

        "rho_physical":
            rho_physical,

        "shots_per_observable":
            int(
                shots_per_observable
            ),

        "number_of_observables":
            len(
                X_STATE_LABELS
            ),

        "total_observable_shots":
            len(
                X_STATE_LABELS
            )
            *
            int(
                shots_per_observable
            ),

        "seed":
            int(
                seed
            )
    }


# ------------------------------------------------------------
# 7. Finite-shot hardware evaluation using reduced tomography
# ------------------------------------------------------------

def evaluate_hardware_candidate_xshot(
    alpha_command,
    phi_command,
    target_concurrence,
    target_phi=0.0,
    shots_per_observable=2000,
    seed=0,
    control_bias=None,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Hardware evaluation using only the seven-observable
    X-state measurement set.
    """

    (
        alpha_target,
        phi_target,
        psi_target
    ) = target_from_concurrence_phase(
        target_concurrence,
        target_phi
    )

    rho_true = (
        prepare_hardware_density_matrix_phase(
            alpha_command=
                alpha_command,

            phi_command=
                phi_command,

            control_bias=
                control_bias,

            noise_after_ry=
                noise_after_ry,

            noise_after_cnot=
                noise_after_cnot
        )
    )

    tomography = (
        run_x_state_tomography(
            rho=rho_true,
            shots_per_observable=
                shots_per_observable,
            seed=seed
        )
    )

    rho_est = tomography[
        "rho_physical"
    ]

    return {
        "alpha":
            float(
                alpha_command
            ),

        "phi":
            wrap_phase(
                phi_command
            ),

        "rho_true":
            rho_true,

        "rho_est":
            rho_est,

        # Controller-visible metrics
        "estimated_concurrence":
            concurrence_mixed(
                rho_est
            ),

        "estimated_fidelity":
            fidelity_with_pure_target(
                rho_est,
                psi_target
            ),

        # Simulator-only diagnostics
        "true_concurrence":
            concurrence_mixed(
                rho_true
            ),

        "true_fidelity":
            fidelity_with_pure_target(
                rho_true,
                psi_target
            ),

        "shots_per_observable":
            int(
                shots_per_observable
            ),

        "seed":
            int(
                seed
            )
    }


# ------------------------------------------------------------
# 8. Reduced-measurement hybrid optimizer
#
# Same search strategy as Cell 8.
# Only the measurement backend changes.
# ------------------------------------------------------------

def run_x_state_hardware_optimizer(
    target_concurrence,
    target_phi=0.0,

    shots_per_observable=2000,
    seed=3,

    n_alpha=17,
    n_phi=25,

    refinement_rounds=2,
    refinement_alpha_points=9,
    refinement_phi_points=13,

    weight_concurrence=1.0,
    weight_fidelity=1.0,

    control_bias=None,
    noise_after_ry=None,
    noise_after_cnot=None
):
    """
    Hybrid hardware optimizer using seven-observable
    X-state tomography instead of full tomography.
    """

    rng = np.random.default_rng(
        seed
    )

    history = []

    best = None

    def evaluate_point(
        alpha,
        phi,
        stage
    ):
        nonlocal best

        shot_seed = int(
            rng.integers(
                0,
                np.iinfo(
                    np.int32
                ).max
            )
        )

        result = (
            evaluate_hardware_candidate_xshot(
                alpha_command=
                    float(
                        alpha
                    ),

                phi_command=
                    float(
                        phi
                    ),

                target_concurrence=
                    target_concurrence,

                target_phi=
                    target_phi,

                shots_per_observable=
                    shots_per_observable,

                seed=
                    shot_seed,

                control_bias=
                    control_bias,

                noise_after_ry=
                    noise_after_ry,

                noise_after_cnot=
                    noise_after_cnot
            )
        )

        loss_C = (
            result[
                "estimated_concurrence"
            ]
            -
            target_concurrence
        ) ** 2

        loss_F = (
            1.0
            -
            result[
                "estimated_fidelity"
            ]
        )

        loss = (
            weight_concurrence
            *
            loss_C
            +
            weight_fidelity
            *
            loss_F
        )

        result.update({
            "loss":
                float(
                    loss
                ),

            "loss_C":
                float(
                    loss_C
                ),

            "loss_F":
                float(
                    loss_F
                ),

            "stage":
                int(
                    stage
                )
        })

        history.append(
            result
        )

        if (
            best is None
            or result["loss"]
            < best["loss"]
        ):
            best = result

    # --------------------------------------------------------
    # Global grid
    # --------------------------------------------------------

    alpha_grid = np.linspace(
        0.0,
        np.pi / 2.0,
        n_alpha
    )

    phi_grid = np.linspace(
        -np.pi,
        np.pi,
        n_phi,
        endpoint=False
    )

    for alpha in alpha_grid:

        for phi in phi_grid:

            evaluate_point(
                alpha,
                phi,
                stage=0
            )

    delta_alpha = (
        alpha_grid[1]
        - alpha_grid[0]
    )

    delta_phi = (
        2.0
        * np.pi
        / n_phi
    )

    # --------------------------------------------------------
    # Refinement
    # --------------------------------------------------------

    for stage in range(
        1,
        refinement_rounds + 1
    ):

        alpha_center = best[
            "alpha"
        ]

        phi_center = best[
            "phi"
        ]

        alpha_grid = np.linspace(
            alpha_center
            - delta_alpha,

            alpha_center
            + delta_alpha,

            refinement_alpha_points
        )

        alpha_grid = np.clip(
            alpha_grid,
            0.0,
            np.pi / 2.0
        )

        phi_grid = np.linspace(
            phi_center
            - delta_phi,

            phi_center
            + delta_phi,

            refinement_phi_points
        )

        phi_grid = np.array([
            wrap_phase(
                phi
            )
            for phi in phi_grid
        ])

        for alpha in alpha_grid:

            for phi in phi_grid:

                evaluate_point(
                    alpha,
                    phi,
                    stage=stage
                )

        delta_alpha = (
            2.0
            * delta_alpha
            /
            (
                refinement_alpha_points
                - 1
            )
        )

        delta_phi = (
            2.0
            * delta_phi
            /
            (
                refinement_phi_points
                - 1
            )
        )

    number_of_evaluations = len(
        history
    )

    total_shots = (
        number_of_evaluations
        *
        len(
            X_STATE_LABELS
        )
        *
        int(
            shots_per_observable
        )
    )

    return {
        "best":
            best,

        "history":
            history,

        "number_of_evaluations":
            number_of_evaluations,

        "shots_per_observable":
            int(
                shots_per_observable
            ),

        "number_of_observables":
            len(
                X_STATE_LABELS
            ),

        "total_shots":
            int(
                total_shots
            ),

        "seed":
            int(
                seed
            )
    }


# ------------------------------------------------------------
# 9. Cell 9 unit tests
# ------------------------------------------------------------

def run_cell9_unit_tests(
    verbose=True
):

    # ========================================================
    # Test 1
    # Current circuit + noise family preserves X-state form.
    #
    # Include nontrivial pre- and post-CNOT noise.
    # ========================================================

    test_configs = [

        (
            0.83,
            0.41,
            {
                "amplitude": (0.04, 0.02),
                "dephasing": (0.07, 0.03),
                "depolarizing": (0.05, 0.09)
            },
            {
                "amplitude": (0.06, 0.08),
                "dephasing": (0.04, 0.05),
                "depolarizing": (0.03, 0.02)
            }
        ),

        (
            1.10,
            -0.73,
            {
                "depolarizing": (0.11, 0.17)
            },
            {
                "amplitude": (0.13, 0.07),
                "dephasing": (0.08, 0.12)
            }
        ),

        (
            np.pi / 2.0,
            1.20,
            {
                "amplitude": (0.10, 0.20)
            },
            {
                "depolarizing": (0.15, 0.05)
            }
        )
    ]

    for (
        alpha,
        phi,
        noise_ry,
        noise_cnot
    ) in test_configs:

        rho = (
            prepare_noisy_density_matrix_phase(
                alpha=alpha,
                phi=phi,
                noise_after_ry=noise_ry,
                noise_after_cnot=noise_cnot
            )
        )

        assert (
            x_state_leakage(
                rho
            )
            <
            1e-12
        )

    # ========================================================
    # Test 2
    # Exact seven-observable reconstruction reproduces
    # the full noisy density matrix.
    # ========================================================

    rho_test = (
        prepare_noisy_density_matrix_phase(
            alpha=0.91,
            phi=0.67,

            noise_after_ry={
                "amplitude":
                    (0.07, 0.03),

                "dephasing":
                    (0.06, 0.04),

                "depolarizing":
                    (0.08, 0.12)
            },

            noise_after_cnot={
                "amplitude":
                    (0.05, 0.09),

                "dephasing":
                    (0.03, 0.07),

                "depolarizing":
                    (0.02, 0.04)
            }
        )
    )

    exact_x = exact_x_state_table(
        rho_test
    )

    rho_x = rho_from_x_expectations(
        exact_x
    )

    assert np.allclose(
        rho_x,
        rho_test,
        atol=1e-12
    )

    # ========================================================
    # Test 3
    # Finite-shot reduced tomography returns
    # a physical density matrix.
    # ========================================================

    tomo = run_x_state_tomography(
        rho_test,
        shots_per_observable=5000,
        seed=12345
    )

    assert_density_matrix(
        tomo[
            "rho_physical"
        ],
        atol=1e-10
    )

    # ========================================================
    # Test 4
    # Reduced measurement count is 7, not 15.
    # ========================================================

    assert (
        len(
            X_STATE_LABELS
        )
        ==
        7
    )

    assert (
        len(
            PAULI_LABELS
        )
        ==
        15
    )

    # ========================================================
    # Benchmark from Cell 8
    # ========================================================

    C_target = 1.0
    phi_target = 0.0

    control_bias = {
        "alpha_offset":
            0.25,

        "phi_offset":
            0.70
    }

    noise_after_cnot = {
        "dephasing": (
            0.20,
            0.0
        )
    }

    alpha_corrected = (
        np.pi / 2.0
        -
        0.25
    )

    phi_corrected = (
        -0.70
    )

    # ========================================================
    # Test 5
    # Reduced-measurement optimizer still recovers
    # the noisy Bell-state ceiling.
    # ========================================================

    result = (
        run_x_state_hardware_optimizer(
            target_concurrence=
                C_target,

            target_phi=
                phi_target,

            shots_per_observable=
                2000,

            seed=
                3,

            n_alpha=
                17,

            n_phi=
                25,

            refinement_rounds=
                2,

            control_bias=
                control_bias,

            noise_after_cnot=
                noise_after_cnot
        )
    )

    best = result[
        "best"
    ]

    assert (
        best[
            "true_fidelity"
        ]
        >
        0.895
    )

    assert abs(
        best[
            "true_concurrence"
        ]
        -
        0.80
    ) < 0.01

    # ========================================================
    # Test 6
    # Recovered controls remain close to physical correction.
    # ========================================================

    assert abs(
        best[
            "alpha"
        ]
        -
        alpha_corrected
    ) < 0.10

    assert abs(
        phase_distance(
            best[
                "phi"
            ],
            phi_corrected
        )
    ) < 0.15

    # ========================================================
    # Test 7
    # Shot accounting is reduced exactly by 7/15
    # relative to full tomography for equal search size.
    # ========================================================

    full_tomography_shots = (
        result[
            "number_of_evaluations"
        ]
        *
        len(
            PAULI_LABELS
        )
        *
        result[
            "shots_per_observable"
        ]
    )

    reduced_shots = (
        result[
            "total_shots"
        ]
    )

    ratio = (
        reduced_shots
        /
        full_tomography_shots
    )

    assert np.isclose(
        ratio,
        7.0 / 15.0,
        atol=1e-15
    )

    # ========================================================
    # Test 8
    # Exact expected shot count for frozen search.
    # ========================================================

    assert (
        result[
            "number_of_evaluations"
        ]
        ==
        659
    )

    assert (
        result[
            "total_shots"
        ]
        ==
        9_226_000
    )

    # ========================================================
    # Report
    # ========================================================

    if verbose:

        reduction = (
            1.0
            -
            7.0 / 15.0
        )

        print(
            "Cell 9 unit tests: PASS"
        )

        print(
            "  ✓ current circuit/noise model preserves X-state structure"
        )

        print(
            "  ✓ seven exact Pauli observables reconstruct the complete X state"
        )

        print(
            "  ✓ reduced finite-shot tomography returns a physical state"
        )

        print(
            "  ✓ measurement set reduced from 15 to 7 observables"
        )

        print(
            "  ✓ reduced-measurement controller still approaches hardware ceiling"
        )

        print(
            "  ✓ recovered controls remain close to calibration correction"
        )

        print(
            "  ✓ shot reduction is exactly 7/15 of full tomography"
        )

        print(
            "  ✓ reduced measurement budget is tracked correctly"
        )

        print()

        print(
            "Compressed-measurement benchmark:"
        )

        print(
            f"  alpha corrected target = "
            f"{alpha_corrected:.6f}"
        )

        print(
            f"  alpha controller found = "
            f"{best['alpha']:.6f}"
        )

        print(
            f"  phi corrected target   = "
            f"{phi_corrected:.6f}"
        )

        print(
            f"  phi controller found   = "
            f"{best['phi']:.6f}"
        )

        print()

        print(
            f"  C estimated            = "
            f"{best['estimated_concurrence']:.6f}"
        )

        print(
            f"  C true                 = "
            f"{best['true_concurrence']:.6f}"
        )

        print(
            f"  F estimated            = "
            f"{best['estimated_fidelity']:.6f}"
        )

        print(
            f"  F true                 = "
            f"{best['true_fidelity']:.6f}"
        )

        print()

        print(
            f"  observables/candidate  = "
            f"{len(X_STATE_LABELS)}"
        )

        print(
            f"  full tomography        = "
            f"{len(PAULI_LABELS)}"
        )

        print(
            f"  evaluations            = "
            f"{result['number_of_evaluations']}"
        )

        print(
            f"  full-tomography shots  = "
            f"{full_tomography_shots:,}"
        )

        print(
            f"  reduced shots          = "
            f"{reduced_shots:,}"
        )

        print(
            f"  measurement reduction  = "
            f"{100.0 * reduction:.1f}%"
        )


# Run immediately
run_cell9_unit_tests()

Cell 9 unit tests: PASS
  ✓ current circuit/noise model preserves X-state structure
  ✓ seven exact Pauli observables reconstruct the complete X state
  ✓ reduced finite-shot tomography returns a physical state
  ✓ measurement set reduced from 15 to 7 observables
  ✓ reduced-measurement controller still approaches hardware ceiling
  ✓ recovered controls remain close to calibration correction
  ✓ shot reduction is exactly 7/15 of full tomography
  ✓ reduced measurement budget is tracked correctly

Compressed-measurement benchmark:
  alpha corrected target = 1.320796
  alpha controller found = 1.349903
  phi corrected target   = -0.700000
  phi controller found   = -0.753982

  C estimated            = 0.805817
  C true                 = 0.799661
  F estimated            = 0.902702
  F true                 = 0.899248

  observables/candidate  = 7
  full tomography        = 15
  evaluations            = 659
  full-tomography shots  = 19,770,000
  reduced shots          = 9,226,000
  measu

In [10]:
# ============================================================
# V2 — Cell 10
# Interactive deep-tech dashboard
#
# Requires frozen Cells 1–9.
#
# Purpose:
#   Turn the validated V2 machinery into a compact
#   engineering demonstrator.
#
# Shows:
#   - target state specification
#   - coherent hardware calibration errors
#   - irreversible dephasing
#   - nominal performance
#   - finite-shot hybrid recovery
#   - exact hardware ceiling
#   - recoverable vs irreducible degradation
#   - measurement budget
#
# Physics is unchanged from Cells 1–9.
# ============================================================

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


# ------------------------------------------------------------
# 1. Dashboard calculation
#
# Separate from UI so it can be unit tested.
# ------------------------------------------------------------

def run_v2_dashboard_case(
    target_concurrence=1.0,
    target_phi=0.0,

    alpha_offset=0.25,
    phi_offset=0.70,

    dephasing_q0=0.20,
    dephasing_q1=0.00,

    shots_per_observable=2000,

    seed=3,

    n_alpha=17,
    n_phi=25,
    refinement_rounds=2
):
    """
    Run one complete V2 engineering case.

    Returns:
        target information
        nominal hardware result
        finite-shot controlled result
        exact hardware ceiling
        recoverability decomposition
        shot budget
    """

    (
        alpha_target,
        phi_target,
        psi_target
    ) = target_from_concurrence_phase(
        target_concurrence,
        target_phi
    )

    control_bias = {
        "alpha_offset":
            float(alpha_offset),

        "phi_offset":
            float(phi_offset)
    }

    noise_after_cnot = {
        "dephasing": (
            float(dephasing_q0),
            float(dephasing_q1)
        )
    }

    # --------------------------------------------------------
    # Nominal operation:
    # issue the ideal commands without compensating for
    # hardware offsets.
    # --------------------------------------------------------

    nominal = evaluate_hardware_exact(
        alpha_command=
            alpha_target,

        phi_command=
            phi_target,

        target_concurrence=
            target_concurrence,

        target_phi=
            phi_target,

        control_bias=
            control_bias,

        noise_after_cnot=
            noise_after_cnot
    )

    # --------------------------------------------------------
    # Exact noisy-hardware optimum:
    # oracle / physical ceiling.
    # --------------------------------------------------------

    ceiling = run_exact_hardware_optimizer(
        target_concurrence=
            target_concurrence,

        target_phi=
            phi_target,

        control_bias=
            control_bias,

        noise_after_cnot=
            noise_after_cnot
    )

    # --------------------------------------------------------
    # Finite-shot hybrid control:
    # seven-observable X-state tomography.
    # --------------------------------------------------------

    controlled_run = run_x_state_hardware_optimizer(
        target_concurrence=
            target_concurrence,

        target_phi=
            phi_target,

        shots_per_observable=
            shots_per_observable,

        seed=
            seed,

        n_alpha=
            n_alpha,

        n_phi=
            n_phi,

        refinement_rounds=
            refinement_rounds,

        control_bias=
            control_bias,

        noise_after_cnot=
            noise_after_cnot
    )

    controlled = controlled_run[
        "best"
    ]

    recovery = build_recovery_summary(
        target_concurrence=
            target_concurrence,

        target_phi=
            phi_target,

        nominal=
            nominal,

        controlled=
            controlled,

        hardware_ceiling=
            ceiling
    )

    return {
        "target_concurrence":
            float(
                target_concurrence
            ),

        "target_alpha":
            float(
                alpha_target
            ),

        "target_phi":
            float(
                phi_target
            ),

        "control_bias":
            control_bias,

        "noise_after_cnot":
            noise_after_cnot,

        "nominal":
            nominal,

        "controlled":
            controlled,

        "ceiling":
            ceiling,

        "recovery":
            recovery,

        "optimizer_run":
            controlled_run,

        "shots_per_observable":
            int(
                shots_per_observable
            ),

        "observables_per_candidate":
            len(
                X_STATE_LABELS
            ),

        "full_tomography_observables":
            len(
                PAULI_LABELS
            ),

        "total_shots":
            controlled_run[
                "total_shots"
            ]
    }


# ------------------------------------------------------------
# 2. Text summary
# ------------------------------------------------------------

def print_v2_case_summary(
    case
):
    """
    Human-readable engineering summary.
    """

    nominal = case[
        "nominal"
    ]

    controlled = case[
        "controlled"
    ]

    ceiling = case[
        "ceiling"
    ]

    recovery = case[
        "recovery"
    ]

    run = case[
        "optimizer_run"
    ]

    alpha_corrected = (
        case[
            "target_alpha"
        ]
        -
        case[
            "control_bias"
        ][
            "alpha_offset"
        ]
    )

    phi_corrected = wrap_phase(
        case[
            "target_phi"
        ]
        -
        case[
            "control_bias"
        ][
            "phi_offset"
        ]
    )

    print(
        "V2 HYBRID QUANTUM–CLASSICAL ENGINEERING RESULT"
    )

    print(
        "=" * 52
    )

    print()

    print(
        "Target:"
    )

    print(
        f"  concurrence            = "
        f"{case['target_concurrence']:.4f}"
    )

    print(
        f"  alpha                  = "
        f"{case['target_alpha']:.4f} rad"
    )

    print(
        f"  phase                  = "
        f"{case['target_phi']:.4f} rad"
    )

    print()

    print(
        "Hardware:"
    )

    print(
        f"  alpha offset           = "
        f"{case['control_bias']['alpha_offset']:+.4f} rad"
    )

    print(
        f"  phase offset           = "
        f"{case['control_bias']['phi_offset']:+.4f} rad"
    )

    print(
        f"  q0 dephasing           = "
        f"{case['noise_after_cnot']['dephasing'][0]:.4f}"
    )

    print(
        f"  q1 dephasing           = "
        f"{case['noise_after_cnot']['dephasing'][1]:.4f}"
    )

    print()

    print(
        "Performance:"
    )

    print(
        f"  nominal fidelity       = "
        f"{nominal['fidelity']:.6f}"
    )

    print(
        f"  controlled fidelity    = "
        f"{controlled['true_fidelity']:.6f}"
    )

    print(
        f"  hardware ceiling       = "
        f"{ceiling['fidelity']:.6f}"
    )

    print(
        f"  ideal fidelity         = "
        f"{1.0:.6f}"
    )

    print()

    print(
        f"  nominal concurrence    = "
        f"{nominal['concurrence']:.6f}"
    )

    print(
        f"  controlled concurrence = "
        f"{controlled['true_concurrence']:.6f}"
    )

    print(
        f"  hardware C ceiling     = "
        f"{ceiling['concurrence']:.6f}"
    )

    print()

    print(
        "Control:"
    )

    print(
        f"  expected alpha command = "
        f"{alpha_corrected:.6f}"
    )

    print(
        f"  controller alpha       = "
        f"{controlled['alpha']:.6f}"
    )

    print(
        f"  expected phase command = "
        f"{phi_corrected:.6f}"
    )

    print(
        f"  controller phase       = "
        f"{controlled['phi']:.6f}"
    )

    print()

    print(
        "Recoverability:"
    )

    print(
        f"  recoverable fidelity   = "
        f"{recovery['recoverable_opportunity']:.6f}"
    )

    print(
        f"  fidelity recovered     = "
        f"{recovery['recovered']:.6f}"
    )

    print(
        f"  recovery fraction      = "
        f"{100.0 * recovery['recovered_fraction']:.1f}%"
    )

    print(
        f"  controller gap         = "
        f"{recovery['controller_gap']:.6f}"
    )

    print(
        f"  irreducible gap        = "
        f"{recovery['irreducible_gap']:.6f}"
    )

    print()

    print(
        "Measurement budget:"
    )

    print(
        f"  observables/candidate  = "
        f"{case['observables_per_candidate']}"
    )

    print(
        f"  full tomography        = "
        f"{case['full_tomography_observables']}"
    )

    print(
        f"  evaluations            = "
        f"{run['number_of_evaluations']}"
    )

    print(
        f"  shots/observable       = "
        f"{case['shots_per_observable']:,}"
    )

    print(
        f"  total shots            = "
        f"{case['total_shots']:,}"
    )


# ------------------------------------------------------------
# 3. Dashboard plots
# ------------------------------------------------------------

def plot_v2_case(
    case
):
    """
    Plot the main V2 engineering results.
    """

    nominal = case[
        "nominal"
    ]

    controlled = case[
        "controlled"
    ]

    ceiling = case[
        "ceiling"
    ]

    # --------------------------------------------------------
    # Fidelity
    # --------------------------------------------------------

    labels = [
        "Nominal",
        "Hybrid\ncontrolled",
        "Hardware\nceiling",
        "Ideal"
    ]

    fidelity_values = [
        nominal[
            "fidelity"
        ],

        controlled[
            "true_fidelity"
        ],

        ceiling[
            "fidelity"
        ],

        1.0
    ]

    plt.figure(
        figsize=(
            7,
            4
        )
    )

    plt.bar(
        labels,
        fidelity_values
    )

    plt.ylim(
        0.0,
        1.05
    )

    plt.ylabel(
        "Target-state fidelity"
    )

    plt.title(
        "Recoverable vs irreducible state degradation"
    )

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.show()

    # --------------------------------------------------------
    # Concurrence
    # --------------------------------------------------------

    concurrence_values = [
        nominal[
            "concurrence"
        ],

        controlled[
            "true_concurrence"
        ],

        ceiling[
            "concurrence"
        ],

        case[
            "target_concurrence"
        ]
    ]

    plt.figure(
        figsize=(
            7,
            4
        )
    )

    plt.bar(
        labels,
        concurrence_values
    )

    plt.ylim(
        0.0,
        1.05
    )

    plt.ylabel(
        "Concurrence"
    )

    plt.title(
        "Entanglement performance"
    )

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.show()

    # --------------------------------------------------------
    # Recoverability budget
    # --------------------------------------------------------

    recovery = case[
        "recovery"
    ]

    budget_labels = [
        "Recovered",
        "Controller\ngap",
        "Irreducible"
    ]

    budget_values = [
        max(
            0.0,
            recovery[
                "recovered"
            ]
        ),

        max(
            0.0,
            recovery[
                "controller_gap"
            ]
        ),

        max(
            0.0,
            recovery[
                "irreducible_gap"
            ]
        )
    ]

    plt.figure(
        figsize=(
            7,
            4
        )
    )

    plt.bar(
        budget_labels,
        budget_values
    )

    plt.ylabel(
        "Fidelity contribution"
    )

    plt.title(
        "Fidelity degradation / recovery budget"
    )

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.show()


# ------------------------------------------------------------
# 4. Unit tests for dashboard calculation
# ------------------------------------------------------------

def run_cell10_unit_tests(
    verbose=True
):

    # Frozen Cell 8 benchmark
    case = run_v2_dashboard_case(
        target_concurrence=
            1.0,

        target_phi=
            0.0,

        alpha_offset=
            0.25,

        phi_offset=
            0.70,

        dephasing_q0=
            0.20,

        dephasing_q1=
            0.00,

        shots_per_observable=
            2000,

        seed=
            3,

        n_alpha=
            17,

        n_phi=
            25,

        refinement_rounds=
            2
    )

    # --------------------------------------------------------
    # Test 1:
    # nominal degradation
    # --------------------------------------------------------

    assert (
        case[
            "nominal"
        ][
            "fidelity"
        ]
        <
        case[
            "ceiling"
        ][
            "fidelity"
        ]
    )

    # --------------------------------------------------------
    # Test 2:
    # known hardware ceiling
    # --------------------------------------------------------

    assert np.isclose(
        case[
            "ceiling"
        ][
            "fidelity"
        ],
        0.90,
        atol=1e-8
    )

    assert np.isclose(
        case[
            "ceiling"
        ][
            "concurrence"
        ],
        0.80,
        atol=1e-7
    )

    # --------------------------------------------------------
    # Test 3:
    # hybrid controller approaches ceiling
    # --------------------------------------------------------

    assert (
        case[
            "controlled"
        ][
            "true_fidelity"
        ]
        >
        0.895
    )

    assert abs(
        case[
            "controlled"
        ][
            "true_concurrence"
        ]
        -
        0.80
    ) < 0.01

    # --------------------------------------------------------
    # Test 4:
    # >95% of recoverable fidelity restored
    # --------------------------------------------------------

    assert (
        case[
            "recovery"
        ][
            "recovered_fraction"
        ]
        >
        0.95
    )

    # --------------------------------------------------------
    # Test 5:
    # irreducible fidelity gap remains 0.10
    # --------------------------------------------------------

    assert np.isclose(
        case[
            "recovery"
        ][
            "irreducible_gap"
        ],
        0.10,
        atol=1e-8
    )

    # --------------------------------------------------------
    # Test 6:
    # seven-observable measurement model
    # --------------------------------------------------------

    assert (
        case[
            "observables_per_candidate"
        ]
        ==
        7
    )

    # --------------------------------------------------------
    # Test 7:
    # frozen search budget
    # --------------------------------------------------------

    assert (
        case[
            "optimizer_run"
        ][
            "number_of_evaluations"
        ]
        ==
        659
    )

    assert (
        case[
            "total_shots"
        ]
        ==
        9_226_000
    )

    if verbose:

        print(
            "Cell 10 unit tests: PASS"
        )

        print(
            "  ✓ dashboard reproduces frozen Cell 8 benchmark"
        )

        print(
            "  ✓ nominal operation lies below hardware ceiling"
        )

        print(
            "  ✓ physical F=0.90 and C=0.80 ceilings recovered"
        )

        print(
            "  ✓ finite-shot controller approaches hardware ceiling"
        )

        print(
            "  ✓ >95% of recoverable fidelity is restored"
        )

        print(
            "  ✓ irreducible decoherence gap remains explicit"
        )

        print(
            "  ✓ seven-observable measurement backend is active"
        )

        print(
            "  ✓ compressed measurement budget is reproduced"
        )


# Run validation immediately
run_cell10_unit_tests()

Cell 10 unit tests: PASS
  ✓ dashboard reproduces frozen Cell 8 benchmark
  ✓ nominal operation lies below hardware ceiling
  ✓ physical F=0.90 and C=0.80 ceilings recovered
  ✓ finite-shot controller approaches hardware ceiling
  ✓ >95% of recoverable fidelity is restored
  ✓ irreducible decoherence gap remains explicit
  ✓ seven-observable measurement backend is active
  ✓ compressed measurement budget is reproduced


In [11]:
# ============================================================
# 5. Interactive V2 dashboard
# ============================================================

def build_v2_optimizer_ui():

    # --------------------------------------------------------
    # Target controls
    # --------------------------------------------------------

    target_C = widgets.FloatSlider(
        value=1.0,
        min=0.0,
        max=1.0,
        step=0.05,
        description="Target concurrence:",
        continuous_update=False,
        readout_format=".2f",
        style={
            "description_width":
                "initial"
        },
        layout=widgets.Layout(
            width="520px"
        )
    )

    target_phi = widgets.FloatSlider(
        value=0.0,
        min=-np.pi,
        max=np.pi,
        step=0.05,
        description="Target phase φ:",
        continuous_update=False,
        readout_format=".2f",
        style={
            "description_width":
                "initial"
        },
        layout=widgets.Layout(
            width="520px"
        )
    )

    # --------------------------------------------------------
    # Coherent hardware errors
    # --------------------------------------------------------

    alpha_offset = widgets.FloatSlider(
        value=0.25,
        min=-0.75,
        max=0.75,
        step=0.05,
        description="α calibration offset:",
        continuous_update=False,
        readout_format=".2f",
        style={
            "description_width":
                "initial"
        },
        layout=widgets.Layout(
            width="520px"
        )
    )

    phi_offset = widgets.FloatSlider(
        value=0.70,
        min=-np.pi,
        max=np.pi,
        step=0.05,
        description="φ calibration offset:",
        continuous_update=False,
        readout_format=".2f",
        style={
            "description_width":
                "initial"
        },
        layout=widgets.Layout(
            width="520px"
        )
    )

    # --------------------------------------------------------
    # Irreversible dephasing
    # --------------------------------------------------------

    dephasing_q0 = widgets.FloatSlider(
        value=0.20,
        min=0.0,
        max=0.80,
        step=0.02,
        description="Qubit 0 dephasing:",
        continuous_update=False,
        readout_format=".2f",
        style={
            "description_width":
                "initial"
        },
        layout=widgets.Layout(
            width="520px"
        )
    )

    dephasing_q1 = widgets.FloatSlider(
        value=0.00,
        min=0.0,
        max=0.80,
        step=0.02,
        description="Qubit 1 dephasing:",
        continuous_update=False,
        readout_format=".2f",
        style={
            "description_width":
                "initial"
        },
        layout=widgets.Layout(
            width="520px"
        )
    )

    # --------------------------------------------------------
    # Measurement settings
    # --------------------------------------------------------

    shots = widgets.IntSlider(
        value=2000,
        min=250,
        max=5000,
        step=250,
        description="Shots / observable:",
        continuous_update=False,
        style={
            "description_width":
                "initial"
        },
        layout=widgets.Layout(
            width="520px"
        )
    )

    seed = widgets.IntText(
        value=3,
        description="Random seed:",
        style={
            "description_width":
                "initial"
        },
        layout=widgets.Layout(
            width="240px"
        )
    )

    # --------------------------------------------------------
    # Search-quality preset
    # --------------------------------------------------------

    search_mode = widgets.Dropdown(
        options=[
            (
                "Fast demo",
                "fast"
            ),
            (
                "Validated benchmark",
                "validated"
            )
        ],

        value="fast",

        description="Search mode:",

        style={
            "description_width":
                "initial"
        },

        layout=widgets.Layout(
            width="320px"
        )
    )

    run_button = widgets.Button(
        description=
            "Run V2 Hybrid Controller",

        button_style=
            "primary",

        layout=widgets.Layout(
            width="240px"
        )
    )

    output = widgets.Output()

    # --------------------------------------------------------
    # Run callback
    # --------------------------------------------------------

    def on_run_clicked(_):

        with output:

            clear_output(
                wait=True
            )

            if (
                search_mode.value
                ==
                "fast"
            ):

                n_alpha = 13
                n_phi = 17
                refinement_rounds = 1

            else:

                n_alpha = 17
                n_phi = 25
                refinement_rounds = 2

            print(
                "Running hybrid controller..."
            )

            print()

            case = run_v2_dashboard_case(
                target_concurrence=
                    target_C.value,

                target_phi=
                    target_phi.value,

                alpha_offset=
                    alpha_offset.value,

                phi_offset=
                    phi_offset.value,

                dephasing_q0=
                    dephasing_q0.value,

                dephasing_q1=
                    dephasing_q1.value,

                shots_per_observable=
                    shots.value,

                seed=
                    seed.value,

                n_alpha=
                    n_alpha,

                n_phi=
                    n_phi,

                refinement_rounds=
                    refinement_rounds
            )

            clear_output(
                wait=True
            )

            print_v2_case_summary(
                case
            )

            print()

            plot_v2_case(
                case
            )

    run_button.on_click(
        on_run_clicked
    )

    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    header = widgets.HTML(
        """
        <h3>
        V2 Hybrid Quantum–Classical Controller
        </h3>

        <p>
        <b>Goal:</b>
        separate recoverable coherent hardware error from
        irreversible quantum-state degradation, then use
        finite-shot classical feedback to approach the
        physical hardware ceiling.
        </p>
        """
    )

    target_box = widgets.VBox([
        widgets.HTML(
            "<b>Target quantum state</b>"
        ),
        target_C,
        target_phi
    ])

    hardware_box = widgets.VBox([
        widgets.HTML(
            "<b>Hardware imperfections</b>"
        ),
        alpha_offset,
        phi_offset,
        dephasing_q0,
        dephasing_q1
    ])

    measurement_box = widgets.VBox([
        widgets.HTML(
            "<b>Hybrid measurement / control</b>"
        ),
        shots,
        widgets.HBox([
            seed,
            search_mode
        ])
    ])

    controls = widgets.VBox([
        header,
        target_box,
        hardware_box,
        measurement_box,
        run_button
    ])

    return widgets.VBox([
        controls,
        output
    ])


# ------------------------------------------------------------
# 6. Launch UI
# ------------------------------------------------------------

v2_ui = build_v2_optimizer_ui()

display(
    v2_ui
)